# Plastic Waste Classification — Full Pipeline

Run each cell in order to execute the complete pipeline from dataset preparation through training, evaluation, and visualisation.

| Cell | Description |
|------|-------------|
| Setup | Fix `__file__` so `Path(__file__).parent` resolves correctly |
| Step 1 | Convert TIP (Pascal VOC XML) to class-sorted folders |
| Step 2 | Convert WaDaBa (filename-encoded) to class-sorted folders |
| Step 3 | Merge into unified dataset + compute class weights |
| Step 0b | *(Optional)* Inject real conveyor crops into training set |
| Step 4 | 3-phase EfficientNet-B3 training (Phase 3 uses SAM optimizer) |
| Step 5 | Evaluate on clean test set — classification report + WaDaBa analysis |
| Step 6 | Grad-CAM visualisation, confidence threshold report, ONNX export |
| Step 7 | Training curves, class distribution, deformation accuracy plots |
| Conveyor | Real-world conveyor belt evaluation with TTA |


---
## Setup — fix __file__ for notebook

In [3]:
# Notebook setup — makes Path(__file__).parent work in all cells below
import os as _os
__file__ = _os.path.abspath("pipeline.ipynb")


---
## Step 1 — Convert TIP Dataset

In [4]:
"""
Step 1: Convert TIP dataset (Pascal VOC XML format) to class-sorted folder structure.

For each split (train / valid / test):
  - Scan dataset/tip/<split>/ for .jpg files
  - Find the matching .xml file (same stem)
  - Parse <object><name> tags to extract class labels
  - Normalise the label to one of: PET, HDPE, LDPE, PP, PS, OTHER
  - Copy the image to dataset/tip_sorted/<split>/<class>/tip_<original_filename>

Run: python step1_convert_tip.py
"""

import os
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict

# ── paths ─────────────────────────────────────────────────────────────────────
BASE_DIR   = Path("/kaggle/working")
SRC_ROOT   = Path("/kaggle/input/datasets/durvibangera/dataset/tip/tip")
DST_ROOT   = BASE_DIR / "dataset" / "tip_sorted"
SPLITS     = ["train", "valid", "test"]
TARGET_CLASSES = {"PET", "HDPE", "LDPE", "PP", "PS", "OTHER"}

# ── class normalisation map ────────────────────────────────────────────────────
RAW_TO_CLASS: dict[str, str] = {
    # PET
    "pet":                            "PET",
    "pete":                           "PET",
    "polyethylene terephthalate":     "PET",
    # HDPE
    "hdpe":                           "HDPE",
    "pe-hd":                          "HDPE",
    "high density polyethylene":      "HDPE",
    # LDPE
    "ldpe":                           "LDPE",
    "pe-ld":                          "LDPE",
    "low density polyethylene":       "LDPE",
    # PP
    "pp":                             "PP",
    "polypropylene":                  "PP",
    # PS
    "ps":                             "PS",
    "polystyrene":                    "PS",
}


def normalise_class(raw: str) -> str:
    """Return one of the six target class names, or 'OTHER'."""
    return RAW_TO_CLASS.get(raw.strip().lower(), "OTHER")


def extract_class_from_xml(xml_path: Path) -> str | None:
    """Return the normalised class name from the first <object><name> in the XML.

    Returns None on any parse error or if no <object> tag is present.
    """
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        obj = root.find("object")
        if obj is None:
            print(f"  [WARN] No <object> tag in {xml_path.name} — skipping")
            return None
        name_tag = obj.find("name")
        if name_tag is None or not name_tag.text:
            print(f"  [WARN] Empty <name> tag in {xml_path.name} — skipping")
            return None
        return normalise_class(name_tag.text)
    except ET.ParseError as exc:
        print(f"  [WARN] Malformed XML {xml_path.name}: {exc} — skipping")
        return None


def process_split(split: str) -> dict[str, int]:
    """Process one data split.  Returns per-class image counts."""
    src_dir = SRC_ROOT / split
    counts: dict[str, int] = defaultdict(int)
    skipped = 0

    if not src_dir.is_dir():
        print(f"  [WARN] Source directory not found: {src_dir}")
        return counts

    jpg_files = sorted(src_dir.glob("*.jpg"))
    print(f"\n  Found {len(jpg_files)} JPG files in {src_dir}")

    for jpg_path in jpg_files:
        xml_path = jpg_path.with_suffix(".xml")

        if not xml_path.exists():
            print(f"  [WARN] No matching XML for {jpg_path.name} — skipping")
            skipped += 1
            continue

        cls = extract_class_from_xml(xml_path)
        if cls is None:
            skipped += 1
            continue

        dst_dir = DST_ROOT / split / cls
        dst_dir.mkdir(parents=True, exist_ok=True)

        dst_name = f"tip_{jpg_path.name}"
        dst_path = dst_dir / dst_name

        shutil.copy2(jpg_path, dst_path)
        counts[cls] += 1

    if skipped:
        print(f"  Skipped {skipped} file(s) in split '{split}'")

    return dict(counts)


def main() -> None:
    print("=" * 60)
    print("Step 1 — TIP Dataset Conversion")
    print("=" * 60)

    grand_total = 0
    all_counts: dict[str, dict[str, int]] = {}

    for split in SPLITS:
        print(f"\n[{split.upper()}]")
        counts = process_split(split)
        all_counts[split] = counts
        total = sum(counts.values())
        grand_total += total

        for cls in sorted(TARGET_CLASSES):
            print(f"  {cls:6s}: {counts.get(cls, 0):5d}")
        print(f"  {'TOTAL':6s}: {total:5d}")

    print("\n" + "=" * 60)
    print(f"Grand total images copied: {grand_total}")
    print(f"Output root: {DST_ROOT}")
    print("=" * 60)


main()


Step 1 — TIP Dataset Conversion

[TRAIN]

  Found 5879 JPG files in /kaggle/input/datasets/durvibangera/dataset/tip/tip/train
  [WARN] No <object> tag in PET-953_jpg.rf.c38c9da427ffe3828055de2bd2972926.xml — skipping
  Skipped 1 file(s) in split 'train'
  HDPE  :  1035
  LDPE  :   637
  OTHER :  1789
  PET   :  1628
  PP    :   525
  PS    :   264
  TOTAL :  5878

[VALID]

  Found 1494 JPG files in /kaggle/input/datasets/durvibangera/dataset/tip/tip/valid
  [WARN] No <object> tag in PET-657_jpg.rf.b4588935fd50690f8a5fca3657bddd8c.xml — skipping
  Skipped 1 file(s) in split 'valid'
  HDPE  :   285
  LDPE  :   161
  OTHER :   384
  PET   :   445
  PP    :   137
  PS    :    81
  TOTAL :  1493

[TEST]

  Found 773 JPG files in /kaggle/input/datasets/durvibangera/dataset/tip/tip/test
  HDPE  :   134
  LDPE  :    78
  OTHER :   235
  PET   :   225
  PP    :    67
  PS    :    34
  TOTAL :   773

Grand total images copied: 8144
Output root: /kaggle/working/dataset/tip_sorted


---
## Step 2 — Convert WaDaBa Dataset

In [5]:
"""
Step 2: Convert WaDaBa dataset to class-sorted folder structure.

WaDaBa filenames encode plastic properties directly:
  Format: {4digits}_{aNN}b{NN}c{N}d{N}e{N}f{N}g{N}h{N}_jpg.rf.{hash}.jpg
  Example: 0029_a01b00c2d0e0f0g0h1_jpg.rf.<hash>.jpg

Encoding:
  a00 or a07 = OTHER
  a01        = PET
  a02        = HDPE
  a03        = OTHER  (PVC — mapped to OTHER)
  a04        = LDPE
  a05        = PP
  a06        = PS
  d{0-3}     = deformation level (0=none … 3=high)
  e{0-3}     = dirtiness level  (0=clean … 3=very dirty)

For each split:
  - Scan dataset/wadaba/<split>/ for .jpg files
  - Parse filename with regex to extract a, d, e parameters
  - Copy to dataset/wadaba_sorted/<split>/<class>/wadaba_<filename>

Also saves: wadaba_metadata.csv
  columns: filename, split, class, deformation_level, dirtiness_level

Run: python step2_convert_wadaba.py
"""

import re
import shutil
import csv
from pathlib import Path
from collections import defaultdict

# ── paths ─────────────────────────────────────────────────────────────────────
BASE_DIR  = Path("/kaggle/working")
SRC_ROOT  = Path("/kaggle/input/datasets/durvibangera/dataset/wadaba/wadaba")
DST_ROOT  = BASE_DIR / "dataset" / "wadaba_sorted"
META_CSV  = BASE_DIR / "wadaba_metadata.csv"
SPLITS    = ["train", "valid", "test"]

TARGET_CLASSES = {"PET", "HDPE", "LDPE", "PP", "PS", "OTHER"}

# ── plastic type mapping (a-parameter, two-digit code → class) ─────────────────
PLASTIC_MAP: dict[str, str] = {
    "00": "OTHER",
    "01": "PET",
    "02": "HDPE",
    "03": "OTHER",   # PVC
    "04": "LDPE",
    "05": "PP",
    "06": "PS",
    "07": "OTHER",
}

# ── regex extracts plastic type (a), deformation (d), dirtiness (e) ───────────
# Matches the parameter block embedded in WaDaBa/Roboflow filenames, e.g.:
#   0029_a01b00c2d0e0f0g0h1_jpg.rf...
#   0004a01b05c2d0e1f0g1h1.jpg  (original pre-Roboflow format)
PARAM_RE = re.compile(r"a(\d{2})b\d+c\d+d(\d)e(\d)")


def parse_filename(name: str) -> tuple[str, int, int] | None:
    """Return (class_name, deformation_level, dirtiness_level) or None if no match."""
    m = PARAM_RE.search(name)
    if not m:
        return None
    a_code = m.group(1)
    d_level = int(m.group(2))
    e_level = int(m.group(3))
    cls = PLASTIC_MAP.get(a_code, "OTHER")
    return cls, d_level, e_level


def process_split(split: str, meta_rows: list) -> dict[str, int]:
    """Process one split; append metadata rows in-place. Returns per-class counts."""
    src_dir = SRC_ROOT / split
    counts: dict[str, int] = defaultdict(int)
    skipped = 0

    if not src_dir.is_dir():
        print(f"  [WARN] Source directory not found: {src_dir}")
        return counts

    jpg_files = sorted(src_dir.glob("*.jpg"))
    print(f"\n  Found {len(jpg_files)} JPG files in {src_dir}")

    for jpg_path in jpg_files:
        result = parse_filename(jpg_path.name)
        if result is None:
            print(f"  [WARN] Cannot parse filename: {jpg_path.name} — skipping")
            skipped += 1
            continue

        cls, d_level, e_level = result

        dst_dir = DST_ROOT / split / cls
        dst_dir.mkdir(parents=True, exist_ok=True)

        dst_name = f"wadaba_{jpg_path.name}"
        dst_path = dst_dir / dst_name
        shutil.copy2(jpg_path, dst_path)

        counts[cls] += 1
        meta_rows.append({
            "filename":          dst_name,
            "split":             split,
            "class":             cls,
            "deformation_level": d_level,
            "dirtiness_level":   e_level,
        })

    if skipped:
        print(f"  Skipped {skipped} file(s) in split '{split}'")

    return dict(counts)


def save_metadata(rows: list) -> None:
    fieldnames = ["filename", "split", "class", "deformation_level", "dirtiness_level"]
    with open(META_CSV, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"\nMetadata CSV saved → {META_CSV}  ({len(rows)} rows)")


def main() -> None:
    print("=" * 60)
    print("Step 2 — WaDaBa Dataset Conversion")
    print("=" * 60)

    meta_rows: list = []
    grand_total = 0

    for split in SPLITS:
        print(f"\n[{split.upper()}]")
        counts = process_split(split, meta_rows)
        total = sum(counts.values())
        grand_total += total

        for cls in sorted(TARGET_CLASSES):
            print(f"  {cls:6s}: {counts.get(cls, 0):5d}")
        print(f"  {'TOTAL':6s}: {total:5d}")

    save_metadata(meta_rows)

    print("\n" + "=" * 60)
    print(f"Grand total images copied: {grand_total}")
    print(f"Output root: {DST_ROOT}")
    print("=" * 60)


main()


Step 2 — WaDaBa Dataset Conversion

[TRAIN]

  Found 1598 JPG files in /kaggle/input/datasets/durvibangera/dataset/wadaba/wadaba/train
  HDPE  :   109
  LDPE  :     0
  OTHER :    31
  PET   :   900
  PP    :   295
  PS    :   263
  TOTAL :  1598

[VALID]

  Found 399 JPG files in /kaggle/input/datasets/durvibangera/dataset/wadaba/wadaba/valid
  HDPE  :    36
  LDPE  :     0
  OTHER :     6
  PET   :   223
  PP    :    70
  PS    :    64
  TOTAL :   399

[TEST]

  Found 200 JPG files in /kaggle/input/datasets/durvibangera/dataset/wadaba/wadaba/test
  HDPE  :    13
  LDPE  :     0
  OTHER :     3
  PET   :   116
  PP    :    35
  PS    :    33
  TOTAL :   200

Metadata CSV saved → /kaggle/working/wadaba_metadata.csv  (2197 rows)

Grand total images copied: 2197
Output root: /kaggle/working/dataset/wadaba_sorted


---
## Step 3 — Create Unified Dataset

In [6]:
"""
Step 3: Merge TIP-sorted and WaDaBa-sorted datasets into a single unified dataset.

Source trees (produced by steps 1 & 2):
  dataset/tip_sorted/{train,valid,test}/{class}/*.jpg
  dataset/wadaba_sorted/{train,valid,test}/{class}/*.jpg

Output:
  dataset/unified/{train,valid,test}/{class}/*.jpg

Also computes balanced class weights for the training split using scikit-learn
and saves them to class_weights.json.

Run: python step3_create_unified.py
"""

import json
import shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# ── paths ─────────────────────────────────────────────────────────────────────
BASE_DIR     = Path("/kaggle/working")
TIP_ROOT     = BASE_DIR / "dataset" / "tip_sorted"
WADABA_ROOT  = BASE_DIR / "dataset" / "wadaba_sorted"
UNIFIED_ROOT = BASE_DIR / "dataset" / "unified"
WEIGHTS_JSON = BASE_DIR / "class_weights.json"

SPLITS         = ["train", "valid", "test"]
TARGET_CLASSES = sorted(["PET", "HDPE", "LDPE", "PP", "PS", "OTHER"])
# ImageFolder sorts class folders alphabetically, so we use the same order.
# Result: ['HDPE', 'LDPE', 'OTHER', 'PET', 'PP', 'PS']


def copy_split(split: str) -> dict[str, dict[str, int]]:
    """Copy images from both source trees into the unified folder.

    Returns per-source, per-class counts: {'tip': {cls: n}, 'wadaba': {cls: n}}
    """
    tip_counts:    dict[str, int] = defaultdict(int)
    wadaba_counts: dict[str, int] = defaultdict(int)

    for source_root, counts, tag in [
        (TIP_ROOT,    tip_counts,    "tip"),
        (WADABA_ROOT, wadaba_counts, "wadaba"),
    ]:
        src_split = source_root / split
        if not src_split.is_dir():
            print(f"  [WARN] Not found: {src_split}")
            continue

        for cls_dir in src_split.iterdir():
            if not cls_dir.is_dir():
                continue
            cls = cls_dir.name
            dst_dir = UNIFIED_ROOT / split / cls
            dst_dir.mkdir(parents=True, exist_ok=True)

            for img_path in cls_dir.glob("*.jpg"):
                shutil.copy2(img_path, dst_dir / img_path.name)
                counts[cls] += 1

    return {"tip": dict(tip_counts), "wadaba": dict(wadaba_counts)}


def compute_weights(train_dir: Path) -> dict[str, float]:
    """Compute sklearn balanced class weights from the unified train split."""
    labels: list[str] = []
    for cls in TARGET_CLASSES:
        cls_dir = train_dir / cls
        if cls_dir.is_dir():
            n = sum(1 for _ in cls_dir.glob("*.jpg"))
            labels.extend([cls] * n)

    if not labels:
        raise RuntimeError("No training images found in unified/train/")

    classes = np.array(TARGET_CLASSES)
    weights = compute_class_weight("balanced", classes=classes, y=np.array(labels))
    return {cls: float(w) for cls, w in zip(TARGET_CLASSES, weights)}


def main() -> None:
    print("=" * 60)
    print("Step 3 — Create Unified Dataset")
    print("=" * 60)

    grand_total = 0

    for split in SPLITS:
        print(f"\n[{split.upper()}]")
        source_counts = copy_split(split)

        tip_c    = source_counts["tip"]
        wadaba_c = source_counts["wadaba"]

        split_total = 0
        header = f"  {'Class':<8} {'TIP':>6} {'WaDaBa':>8} {'Total':>7}"
        print(header)
        print("  " + "-" * (len(header) - 2))

        for cls in TARGET_CLASSES:
            t = tip_c.get(cls, 0)
            w = wadaba_c.get(cls, 0)
            tot = t + w
            split_total += tot
            print(f"  {cls:<8} {t:>6} {w:>8} {tot:>7}")

        print(f"  {'TOTAL':<8} {sum(tip_c.values()):>6} {sum(wadaba_c.values()):>8} {split_total:>7}")
        grand_total += split_total

    # Compute and save class weights ──────────────────────────────────────────
    print("\nComputing balanced class weights …")
    weights = compute_weights(UNIFIED_ROOT / "train")

    with open(WEIGHTS_JSON, "w") as fh:
        json.dump(weights, fh, indent=2)

    # Report ──────────────────────────────────────────────────────────────────
    print("\nClass weights (balanced):")
    for cls, w in sorted(weights.items(), key=lambda x: -x[1]):
        print(f"  {cls:<8}: {w:.4f}")

    weight_values = list(weights.values())
    ratio = max(weight_values) / min(weight_values)
    most_over  = max(weights, key=lambda k: weights[k])
    most_under = min(weights, key=lambda k: weights[k])
    print(f"\nMost over-weighted (rare)  class: {most_over}  ({weights[most_over]:.4f})")
    print(f"Most under-weighted (common) class: {most_under}  ({weights[most_under]:.4f})")
    print(f"Imbalance ratio: {ratio:.2f}x")

    print("\n" + "=" * 60)
    print(f"Grand total images: {grand_total}")
    print(f"Unified dataset: {UNIFIED_ROOT}")
    print(f"Class weights:   {WEIGHTS_JSON}")
    print("=" * 60)


main()


Step 3 — Create Unified Dataset

[TRAIN]
  Class       TIP   WaDaBa   Total
  --------------------------------
  HDPE       1035      109    1144
  LDPE        637        0     637
  OTHER      1789       31    1820
  PET        1628      900    2528
  PP          525      295     820
  PS          264      263     527
  TOTAL      5878     1598    7476

[VALID]
  Class       TIP   WaDaBa   Total
  --------------------------------
  HDPE        285       36     321
  LDPE        161        0     161
  OTHER       384        6     390
  PET         445      223     668
  PP          137       70     207
  PS           81       64     145
  TOTAL      1493      399    1892

[TEST]
  Class       TIP   WaDaBa   Total
  --------------------------------
  HDPE        134       13     147
  LDPE         78        0      78
  OTHER       235        3     238
  PET         225      116     341
  PP           67       35     102
  PS           34       33      67
  TOTAL       773      200     9

---
## Step 0b — Prepare Conveyor Crops (optional, run before Step 4)

In [7]:
"""
Conveyor Crop Extraction + Dataset Preparation
===============================================
Run this ONCE before step4_train.py to inject real-world conveyor belt crops
into the unified training set.

What this does:
  1. Parses every labeled XML in dataset/conveyor/labels/
  2. Crops each bounding box from the corresponding image
  3. Saves the crop into dataset/unified/train/<class>/conv_<img>_<i>.jpg
  4. Recalculates and overwrites class_weights.json based on the new counts

Safety:
  - Only writes to dataset/unified/train/  — never touches test or valid
  - Skips images/XMLs that are already extracted (idempotent — safe to re-run)
  - Never touches dataset/conveyor/ source files

Run: python step_prepare_conveyor.py
"""

import json
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

from PIL import Image
from tqdm import tqdm

# ── paths ──────────────────────────────────────────────────────────────────────
BASE_DIR        = Path("/kaggle/working")
CONVEYOR_IMG    = Path("/kaggle/input/datasets/durvibangera/dataset/conveyor/conveyor/img")
CONVEYOR_LABELS = Path("/kaggle/input/datasets/durvibangera/dataset/conveyor/conveyor/labels")
UNIFIED_TRAIN   = BASE_DIR / "dataset" / "unified" / "train"
WEIGHTS_JSON    = BASE_DIR / "class_weights.json"

CLASS_NAMES = sorted(["HDPE", "LDPE", "OTHER", "PET", "PP", "PS"])

CONVEYOR_CLASS_MAP = {
    "mix pp":    "PP",
    "mix hd":    "HDPE",
    "mix hdpe":  "HDPE",
    "mix pet":   "PET",
    "mix rigid": "OTHER",
}

# Minimum crop dimension in pixels — skip degenerate boxes
MIN_CROP_PX = 8


def find_xml(img_path: Path) -> Path | None:
    candidate = CONVEYOR_LABELS / (img_path.stem + ".xml")
    if candidate.exists():
        return candidate
    for xml in CONVEYOR_LABELS.glob("*.xml"):
        if xml.stem.lower() == img_path.stem.lower():
            return xml
    return None


def parse_xml(xml_path: Path) -> list[dict]:
    try:
        root = ET.parse(xml_path).getroot()
    except ET.ParseError as e:
        print(f"  [WARN] Malformed XML skipped: {xml_path.name} ({e})")
        return []

    objects = []
    for obj in root.findall("object"):
        raw = (obj.findtext("name") or "").strip().lower()
        cls = CONVEYOR_CLASS_MAP.get(raw)
        if cls is None:
            continue
        bb = obj.find("bndbox")
        if bb is None:
            continue
        try:
            box = (
                int(float(bb.findtext("xmin"))),
                int(float(bb.findtext("ymin"))),
                int(float(bb.findtext("xmax"))),
                int(float(bb.findtext("ymax"))),
            )
        except (TypeError, ValueError):
            continue
        objects.append({"class": cls, "box": box})
    return objects


def extract_crops() -> Counter:
    """Extract all crops and return count of crops written per class."""
    written: Counter = Counter()
    skipped_exist = 0

    img_files = sorted(CONVEYOR_IMG.glob("*.jpg")) + \
                sorted(CONVEYOR_IMG.glob("*.jpeg")) + \
                sorted(CONVEYOR_IMG.glob("*.JPG")) + \
                sorted(CONVEYOR_IMG.glob("*.JPEG"))

    # deduplicate by lowercase stem
    seen: set[str] = set()
    unique: list[Path] = []
    for p in img_files:
        if p.stem.lower() not in seen:
            seen.add(p.stem.lower())
            unique.append(p)

    print(f"Found {len(unique)} labeled conveyor images.")

    for img_path in tqdm(unique, desc="Extracting crops"):
        xml_path = find_xml(img_path)
        if xml_path is None:
            print(f"  [WARN] No XML for {img_path.name}, skipping.")
            continue

        annotations = parse_xml(xml_path)
        if not annotations:
            continue

        try:
            pil_img = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"  [WARN] Cannot open {img_path.name}: {e}")
            continue

        w, h = pil_img.size

        for i, ann in enumerate(annotations):
            cls = ann["class"]
            xmin, ymin, xmax, ymax = ann["box"]

            # Clamp to image bounds
            xmin = max(0, min(xmin, w - 1))
            xmax = max(xmin + 1, min(xmax, w))
            ymin = max(0, min(ymin, h - 1))
            ymax = max(ymin + 1, min(ymax, h))

            # Skip degenerate crops
            if (xmax - xmin) < MIN_CROP_PX or (ymax - ymin) < MIN_CROP_PX:
                continue

            out_dir  = UNIFIED_TRAIN / cls
            out_dir.mkdir(parents=True, exist_ok=True)
            out_name = f"conv_{img_path.stem}_{i:03d}.jpg"
            out_path = out_dir / out_name

            # Idempotent — skip if already written
            if out_path.exists():
                skipped_exist += 1
                continue

            crop = pil_img.crop((xmin, ymin, xmax, ymax))
            crop.save(out_path, quality=95)
            written[cls] += 1

    if skipped_exist:
        print(f"  (skipped {skipped_exist} already-existing crops)")

    return written


def recalculate_weights() -> None:
    """
    Recount all training images per class and rewrite class_weights.json
    using inverse-frequency weighting normalised so the mean weight == 1.
    """
    counts: dict[str, int] = {}
    for cls in CLASS_NAMES:
        cls_dir = UNIFIED_TRAIN / cls
        if cls_dir.exists():
            counts[cls] = len(list(cls_dir.iterdir()))
        else:
            counts[cls] = 0

    total = sum(counts.values())
    n_classes = len(CLASS_NAMES)

    # inv-freq weight: (total / n_classes) / count  →  mean weight = 1
    weights = {
        cls: (total / n_classes) / max(counts[cls], 1)
        for cls in CLASS_NAMES
    }

    WEIGHTS_JSON.write_text(json.dumps(weights, indent=2))

    print("\nUpdated class_weights.json:")
    print(f"  {'Class':<8} {'Count':>8}  {'Weight':>8}")
    print("  " + "-" * 28)
    for cls in CLASS_NAMES:
        print(f"  {cls:<8} {counts[cls]:>8}  {weights[cls]:>8.4f}")
    print(f"\n  Total training images: {total}")


def main() -> None:
    print("=" * 55)
    print("  Conveyor Crop Extraction")
    print("=" * 55)

    written = extract_crops()

    print("\nCrops written this run:")
    if written:
        for cls, n in sorted(written.items()):
            print(f"  {cls}: +{n}")
        print(f"  Total new crops: {sum(written.values())}")
    else:
        print("  None (all already present — dataset is up to date)")

    print("\nRecalculating class weights...")
    recalculate_weights()

    print("\nDone. You can now run: python step4_train.py")


main()


  Conveyor Crop Extraction
Found 95 labeled conveyor images.


Extracting crops:  51%|█████     | 48/95 [00:00<00:00, 102.40it/s]

  [WARN] No XML for IN-SURAT-EV-01-03-2025-VU0001_20250425_160651_Color_Mix_HDPE_rigid.jpeg, skipping.


Extracting crops: 100%|██████████| 95/95 [00:00<00:00, 109.09it/s]

  [WARN] No XML for IN-UMBERGAUM-LP-24-09-2025-VU0001_20260127_110138_ - Copy.jpeg, skipping.

Crops written this run:
  HDPE: +38
  OTHER: +101
  PET: +187
  PP: +136
  Total new crops: 462

Recalculating class weights...

Updated class_weights.json:
  Class       Count    Weight
  ----------------------------
  HDPE         1182    1.1193
  LDPE          637    2.0769
  OTHER        1921    0.6887
  PET          2715    0.4873
  PP            956    1.3839
  PS            527    2.5104

  Total training images: 7938

Done. You can now run: python step4_train.py


---
## Step 4 — Train (3-Phase EfficientNet-B3)

In [9]:
"""
Step 4: Three-phase EfficientNet-B3 training pipeline.

Phase 1 — WaDaBa pre-training (frozen backbone)
  Dataset : dataset/wadaba_sorted/
  Epochs  : 15
  Optim   : Adam lr=1e-3
  Sched   : StepLR(step_size=5, gamma=0.5)
  Saves   : checkpoints/phase1_best.pth, logs/phase1_log.csv

Phase 2 — Unified fine-tuning (last 2 feature blocks unfrozen)
  Dataset : dataset/unified/
  Epochs  : 25
  Optim   : Adam lr=1e-4
  Sched   : StepLR(step_size=8, gamma=0.3)
  Saves   : checkpoints/phase2_best.pth, logs/phase2_log.csv

Phase 3 — Full fine-tuning (all layers unfrozen, early stopping)
  Dataset : dataset/unified/
  Epochs  : 30  (+ early stopping, patience=5 on val loss)
  Optim   : SAM(base=Adam) lr=1e-5, rho=0.05
  Sched   : CosineAnnealingLR(T_max=30)
  Saves   : checkpoints/phase3_best.pth, logs/phase3_log.csv

Hardware: GPU required (asserts torch.cuda.is_available()).

Run: python step4_train.py
"""

import csv
import json
import os
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from torchvision.datasets import ImageFolder

import albumentations as A
from albumentations.pytorch import ToTensorV2

try:
    from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
    def build_backbone() -> nn.Module:
        return efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
except ImportError:
    import torchvision
    def build_backbone() -> nn.Module:          # type: ignore[misc]
        return torchvision.models.efficientnet_b3(pretrained=True)

from tqdm import tqdm

# ── GPU guard ─────────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), (
    "CUDA GPU not found.  This pipeline requires a CUDA-capable GPU."
)
DEVICE = torch.device("cuda")

# ── paths ─────────────────────────────────────────────────────────────────────
BASE_DIR     = Path("/kaggle/working")
WADABA_DIR   = BASE_DIR / "dataset" / "wadaba_sorted"
UNIFIED_DIR  = BASE_DIR / "dataset" / "unified"
CKPT_DIR     = BASE_DIR / "checkpoints"
LOG_DIR      = BASE_DIR / "logs"
WEIGHTS_JSON = BASE_DIR / "class_weights.json"

CKPT_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)

# ── constants ──────────────────────────────────────────────────────────────────
NUM_CLASSES  = 6
IMG_SIZE     = 300
NUM_WORKERS  = 4
PIN_MEMORY   = True

# ImageFolder sorts class names alphabetically:
# ['HDPE', 'LDPE', 'OTHER', 'PET', 'PP', 'PS']  (indices 0-5)
CLASS_NAMES = sorted(["HDPE", "LDPE", "OTHER", "PET", "PP", "PS"])

# ── Albumentations transforms ─────────────────────────────────────────────────
# Training: CLAHE applied randomly (p=0.5) as augmentation alongside other transforms
train_transforms = A.Compose([
    A.Resize(300, 300),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=45),
    A.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.4, hue=0.1),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.ToGray(p=0.15),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# Validation / test: CLAHE applied always (p=1.0) as preprocessing, no augmentation
val_transforms = A.Compose([
    A.Resize(300, 300),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])


# ── Albumentations dataset wrapper ────────────────────────────────────────────
class AlbumentationsDataset(Dataset):
    """Wraps an ImageFolder so Albumentations receives numpy arrays (not PIL)."""
    def __init__(self, image_folder_dataset, transform=None):
        self.dataset = image_folder_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        image = np.array(image)  # PIL → numpy for Albumentations
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
        return image, label


# ── focal loss ───────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    Focal Loss: FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    gamma=2.0 (standard).  Class weights serve as alpha, same as weighted CE.
    Down-weights easy examples so training focuses on hard / minority cases.
    """
    def __init__(self, weight: torch.Tensor | None = None, gamma: float = 2.0) -> None:
        super().__init__()
        self.weight = weight
        self.gamma  = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # Per-sample CE loss (unweighted first to get p_t)
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)                          # probability of correct class
        focal = ((1.0 - pt) ** self.gamma) * ce      # down-weight easy examples
        return focal.mean()

# ── SAM (Sharpness-Aware Minimization) optimizer ────────────────────────────────
class SAM(torch.optim.Optimizer):
    """
    Sharpness-Aware Minimization (SAM) optimizer.

    Finds flatter loss minima than Adam, which generalise better to
    out-of-distribution data (e.g. real conveyor images vs. clean training set).

    Two-step update per batch:
      1. first_step  — perturb weights to local worst-case (rho-ball ascent)
      2. second_step — restore original weights, apply base-optimizer update
                       using the gradient computed at the perturbed point

    Reference: Foret et al. 2021 — https://arxiv.org/abs/2010.01412
    """

    def __init__(
        self, params, base_optimizer, rho: float = 0.05, **kwargs
    ) -> None:
        assert rho >= 0.0, f"Invalid rho: {rho}"
        super().__init__(params, dict(rho=rho, **kwargs))
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups   = self.base_optimizer.param_groups  # shared reference

    @torch.no_grad()
    def first_step(self, zero_grad: bool = False) -> None:
        """Perturb weights toward the sharpest loss direction."""
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None:
                    continue
                self.state[p]["old_p"] = p.data.clone()
                p.add_(p.grad * scale)              # ascent step
        if zero_grad:
            self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad: bool = False) -> None:
        """Restore original weights and apply base-optimizer gradient descent."""
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                p.data = self.state[p]["old_p"]     # restore
        self.base_optimizer.step()
        if zero_grad:
            self.zero_grad()

    def step(self, closure=None) -> None:            # type: ignore[override]
        raise NotImplementedError(
            "Call first_step() then second_step() explicitly; "
            "do not call step() on SAM."
        )

    def _grad_norm(self) -> torch.Tensor:
        device = self.param_groups[0]["params"][0].device
        return torch.norm(
            torch.stack([
                p.grad.norm(p=2).to(device)
                for group in self.param_groups
                for p in group["params"]
                if p.grad is not None
            ]),
            p=2,
        )

    def load_state_dict(self, state_dict) -> None:
        super().load_state_dict(state_dict)
        self.base_optimizer.param_groups = self.param_groups

# ── model construction ────────────────────────────────────────────────────────
def build_model() -> nn.Module:
    """Return EfficientNet-B3 with a custom 6-class head."""
    model = build_backbone()
    # Replace the default classifier (1536→1000) with our head
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(1536, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.2),
        nn.Linear(512, NUM_CLASSES),
    )
    return model.to(DEVICE)


# ── freeze / unfreeze helpers ─────────────────────────────────────────────────
def freeze_backbone(model: nn.Module) -> None:
    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True


def unfreeze_last_two_blocks(model: nn.Module) -> None:
    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.features[-1].parameters():
        param.requires_grad = True
    for param in model.features[-2].parameters():
        param.requires_grad = True
    for param in model.classifier.parameters():
        param.requires_grad = True


def unfreeze_all(model: nn.Module) -> None:
    for param in model.parameters():
        param.requires_grad = True


# ── data loaders ──────────────────────────────────────────────────────────────
def make_loaders(
    root: Path,
    batch_size: int,
) -> tuple[DataLoader, DataLoader]:
    # Load raw PIL images (no torchvision transform), then wrap with Albumentations
    base_train = ImageFolder(str(root / "train"))
    base_val   = ImageFolder(str(root / "valid"))

    train_ds = AlbumentationsDataset(base_train, transform=train_transforms)
    val_ds   = AlbumentationsDataset(base_val, transform=val_transforms)

    # ── WeightedRandomSampler ─────────────────────────────────────────────────
    # Gives each class equal expected frequency per batch, independent of
    # dataset size.  This works alongside FocalLoss (they address different
    # aspects: sampling frequency vs. gradient magnitude per example).
    class_counts = [0] * NUM_CLASSES
    for _, lbl in base_train.samples:
        class_counts[lbl] += 1
    sample_weights = [
        1.0 / class_counts[lbl] for _, lbl in base_train.samples
    ]
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, sampler=sampler,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )
    return train_loader, val_loader


def load_class_weights() -> torch.Tensor:
    """Load balanced class weights into a CUDA tensor (alphabetical order)."""
    with open(WEIGHTS_JSON) as fh:
        w_dict: dict[str, float] = json.load(fh)
    return torch.tensor(
        [w_dict[cls] for cls in CLASS_NAMES], dtype=torch.float32
    ).to(DEVICE)


# ── CSV logger ────────────────────────────────────────────────────────────────
class CsvLogger:
    def __init__(self, path: Path) -> None:
        self.path = path
        with open(path, "w", newline="") as fh:
            csv.writer(fh).writerow(
                ["epoch", "train_loss", "train_acc", "val_loss", "val_acc", "lr"]
            )

    def log(self, epoch: int, tl: float, ta: float,
            vl: float, va: float, lr: float) -> None:
        with open(self.path, "a", newline="") as fh:
            csv.writer(fh).writerow(
                [epoch, f"{tl:.6f}", f"{ta:.4f}",
                 f"{vl:.6f}", f"{va:.4f}", f"{lr:.2e}"]
            )


# ── one epoch ─────────────────────────────────────────────────────────────────
def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer | None,
    training: bool,
) -> tuple[float, float]:
    """Return (avg_loss, accuracy)."""
    model.train(training)
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.set_grad_enabled(training):
        for imgs, labels in tqdm(loader, desc="  train" if training else "  val  ",
                                 leave=False, dynamic_ncols=True):
            imgs   = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            logits = model(imgs)
            loss   = criterion(logits, labels)

            if training and optimizer is not None:
                if isinstance(optimizer, SAM):
                    # SAM needs two forward/backward passes per batch:
                    # pass 1 — compute gradient, perturb weights to worst-case
                    loss.backward()
                    optimizer.first_step(zero_grad=True)
                    # pass 2 — compute gradient at perturbed point, then restore
                    criterion(model(imgs), labels).backward()
                    optimizer.second_step(zero_grad=True)
                else:
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += imgs.size(0)

    return total_loss / total, correct / total


# ── training phase ────────────────────────────────────────────────────────────
def train_phase(
    phase_num:  int,
    model:      nn.Module,
    train_loader: DataLoader,
    val_loader:   DataLoader,
    criterion:  nn.Module,
    optimizer:  torch.optim.Optimizer,
    scheduler,
    num_epochs: int,
    ckpt_path:  Path,
    log_path:   Path,
    early_stop_patience: int | None = None,
) -> None:
    print(f"\n{'='*60}")
    print(f"Phase {phase_num}  —  {num_epochs} epochs")
    print(f"{'='*60}")

    logger     = CsvLogger(log_path)
    best_val   = float("inf")
    no_improve = 0

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = run_epoch(
            model, train_loader, criterion, optimizer, training=True
        )
        val_loss, val_acc = run_epoch(
            model, val_loader, criterion, None, training=False
        )

        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step()

        logger.log(epoch, train_loss, train_acc, val_loss, val_acc, current_lr)

        print(
            f"  Epoch {epoch:3d}/{num_epochs}  "
            f"train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  "
            f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  lr={current_lr:.2e}"
        )

        # Save best checkpoint
        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), ckpt_path)
            print(f"    ✓ Best model saved (val_loss={best_val:.4f})")
            no_improve = 0
        else:
            no_improve += 1

        # Early stopping
        if early_stop_patience is not None and no_improve >= early_stop_patience:
            print(f"  Early stopping triggered (no improvement for "
                  f"{early_stop_patience} epochs)")
            break

    print(f"Phase {phase_num} complete.  Best val_loss={best_val:.4f}")


# ── main ──────────────────────────────────────────────────────────────────────
def main() -> None:
    cw = load_class_weights()
    # FocalLoss replaces plain CrossEntropyLoss.
    # Class weights act as alpha (handles inter-class frequency imbalance).
    # gamma=2.0 further concentrates gradient on hard / minority examples.
    criterion = FocalLoss(weight=cw, gamma=2.0)

    # ── Phase 1: WaDaBa only, frozen backbone ─────────────────────────────────
    model = build_model()
    freeze_backbone(model)

    train_loader1, val_loader1 = make_loaders(WADABA_DIR, batch_size=32)
    opt1   = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    sched1 = StepLR(opt1, step_size=5, gamma=0.5)

    train_phase(
        phase_num=1,
        model=model,
        train_loader=train_loader1,
        val_loader=val_loader1,
        criterion=criterion,
        optimizer=opt1,
        scheduler=sched1,
        num_epochs=10,
        ckpt_path=CKPT_DIR / "phase1_best.pth",
        log_path=LOG_DIR / "phase1_log.csv",
    )

    # ── Phase 2: Unified, last 2 blocks unfrozen ──────────────────────────────
    model.load_state_dict(torch.load(CKPT_DIR / "phase1_best.pth"))
    unfreeze_last_two_blocks(model)

    train_loader2, val_loader2 = make_loaders(UNIFIED_DIR, batch_size=32)
    opt2   = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
    sched2 = StepLR(opt2, step_size=8, gamma=0.3)

    train_phase(
        phase_num=2,
        model=model,
        train_loader=train_loader2,
        val_loader=val_loader2,
        criterion=criterion,
        optimizer=opt2,
        scheduler=sched2,
        num_epochs=15,
        ckpt_path=CKPT_DIR / "phase2_best.pth",
        log_path=LOG_DIR / "phase2_log.csv",
    )

    # ── Phase 3: Unified, full fine-tuning, SAM + early stopping ─────────────
    # SAM finds flatter minima → better generalisation to real conveyor images.
    model.load_state_dict(torch.load(CKPT_DIR / "phase2_best.pth"))
    unfreeze_all(model)

    train_loader3, val_loader3 = make_loaders(UNIFIED_DIR, batch_size=16)
    opt3   = SAM(model.parameters(), base_optimizer=Adam, lr=1e-5, rho=0.05)
    sched3 = CosineAnnealingLR(opt3, T_max=30)

    train_phase(
        phase_num=3,
        model=model,
        train_loader=train_loader3,
        val_loader=val_loader3,
        criterion=criterion,
        optimizer=opt3,
        scheduler=sched3,
        num_epochs=20,
        ckpt_path=CKPT_DIR / "phase3_best.pth",
        log_path=LOG_DIR  / "phase3_log.csv",
        early_stop_patience=5,
    )

    print("\nAll three phases complete.")
    print(f"Final checkpoint: {CKPT_DIR / 'phase3_best.pth'}")


main()

/tmp/ipykernel_55/3166236413.py:96: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),



Phase 1  —  10 epochs


  Epoch   1/10  train_loss=0.6349  train_acc=0.5150  val_loss=0.1619  val_acc=0.7569  lr=1.00e-03
    ✓ Best model saved (val_loss=0.1619)


  Epoch   2/10  train_loss=0.3586  train_acc=0.6752  val_loss=0.1997  val_acc=0.7143  lr=1.00e-03


  Epoch   3/10  train_loss=0.3094  train_acc=0.6815  val_loss=0.1387  val_acc=0.8070  lr=1.00e-03
    ✓ Best model saved (val_loss=0.1387)


...

  Epoch  11/20  train_loss=0.0431  train_acc=0.9543  val_loss=0.0318  val_acc=0.9392  lr=7.50e-06


  Epoch  12/20  train_loss=0.0409  train_acc=0.9577  val_loss=2.0912  val_acc=0.9212  lr=7.03e-06


  Epoch  13/20  train_loss=0.0367  train_acc=0.9540  val_loss=0.0933  val_acc=0.9117  lr=6.55e-06


  Epoch  14/20  train_loss=0.0355  train_acc=0.9599  val_loss=0.0478  val_acc=0.9271  lr=6.04e-06


  Epoch  15/20  train_loss=0.0349  train_acc=0.9594  val_loss=0.1228  val_acc=0.8811  lr=5.52e-06
  Early stopping triggered (no improvement for 5 epochs)
Phase 3 complete.  Best val_loss=0.0271

All three phases complete.
Final checkpoint: /kaggle/working/checkpoints/phase3_best.pth


---
## Step 5 — Evaluate

In [10]:
"""
Step 5: Evaluate the trained model on the test split.

Loads checkpoints/phase3_best.pth and runs inference on dataset/unified/test/.

Outputs (all in results/):
  confusion_matrix.png          — colour-coded normalised confusion matrix
  evaluation_report.txt         — sklearn classification report + WaDaBa group analysis
  deformation_results.json      — {d0: acc, d1: acc, d2: acc, d3: acc}
  dirtiness_results.json        — {e0: acc, e1: acc, e2: acc, e3: acc}

Hardware: GPU required.

Run: python step5_evaluate.py
"""

import json
import os
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

try:
    from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
    def build_backbone() -> nn.Module:
        return efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
except ImportError:
    import torchvision
    def build_backbone() -> nn.Module:          # type: ignore[misc]
        return torchvision.models.efficientnet_b3(pretrained=False)

# ── GPU guard ─────────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), "CUDA GPU is required for evaluation."
DEVICE = torch.device("cuda")

# ── paths ─────────────────────────────────────────────────────────────────────
BASE_DIR   = Path(__file__).parent
CKPT_PATH  = BASE_DIR / "checkpoints" / "phase3_best.pth"
TEST_DIR   = BASE_DIR / "dataset" / "unified" / "test"
META_CSV   = BASE_DIR / "wadaba_metadata.csv"
RESULTS    = BASE_DIR / "results"
RESULTS.mkdir(exist_ok=True)

NUM_CLASSES = 6
IMG_SIZE    = 300
NUM_WORKERS = 4
BATCH_SIZE  = 32

CLASS_NAMES = sorted(["HDPE", "LDPE", "OTHER", "PET", "PP", "PS"])

VAL_TF = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


# ── model ─────────────────────────────────────────────────────────────────────
def load_model() -> nn.Module:
    model = build_backbone()
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(1536, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.2),
        nn.Linear(512, NUM_CLASSES),
    )
    state = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(state)
    model.to(DEVICE).eval()
    return model


# ── dataset that also returns image paths ─────────────────────────────────────
class ImageFolderWithPaths(ImageFolder):
    """Extends ImageFolder to return (image, label, path) tuples."""
    def __getitem__(self, index):
        img, label = super().__getitem__(index)
        path = self.samples[index][0]
        return img, label, path


# ── inference ─────────────────────────────────────────────────────────────────
def run_inference(model: nn.Module) -> tuple[list[int], list[int], list[str]]:
    """Returns (all_preds, all_labels, all_paths)."""
    ds = ImageFolderWithPaths(str(TEST_DIR), transform=VAL_TF)
    loader = DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )

    all_preds:  list[int] = []
    all_labels: list[int] = []
    all_paths:  list[str] = []

    with torch.no_grad():
        for imgs, labels, paths in loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            logits = model(imgs)
            preds  = logits.argmax(1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.tolist())
            all_paths.extend(paths)

    return all_preds, all_labels, all_paths


# ── confusion matrix plot ─────────────────────────────────────────────────────
def plot_confusion_matrix(y_true: list[int], y_pred: list[int]) -> None:
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, ax = plt.subplots(figsize=(8, 6))
    img = ax.imshow(cm_norm, interpolation="nearest", cmap="Blues",
                    vmin=0.0, vmax=1.0)
    plt.colorbar(img, ax=ax)

    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("Normalised Confusion Matrix")

    thresh = 0.5
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, f"{cm_norm[i, j]:.2f}",
                    ha="center", va="center",
                    color="white" if cm_norm[i, j] > thresh else "black",
                    fontsize=8)

    fig.tight_layout()
    out = RESULTS / "confusion_matrix.png"
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"Confusion matrix saved → {out}")


# ── WaDaBa deformation / dirtiness analysis ──────────────────────────────────
def wadaba_group_analysis(
    all_preds:  list[int],
    all_labels: list[int],
    all_paths:  list[str],
) -> tuple[dict[str, float], dict[str, float]]:
    """Return per-deformation and per-dirtiness accuracy dicts."""
    if not META_CSV.exists():
        print(f"[WARN] {META_CSV} not found — skipping WaDaBa group analysis")
        return {}, {}

    meta_df = pd.read_csv(META_CSV)
    # Build lookup: filename → (deformation_level, dirtiness_level)
    meta_lookup: dict[str, tuple[int, int]] = {
        row["filename"]: (int(row["deformation_level"]), int(row["dirtiness_level"]))
        for _, row in meta_df.iterrows()
    }

    deform_correct:  dict[int, int] = defaultdict(int)
    deform_total:    dict[int, int] = defaultdict(int)
    dirty_correct:   dict[int, int] = defaultdict(int)
    dirty_total:     dict[int, int] = defaultdict(int)

    for pred, label, path in zip(all_preds, all_labels, all_paths):
        fname = os.path.basename(path)
        if fname not in meta_lookup:
            continue
        d_level, e_level = meta_lookup[fname]
        correct = int(pred == label)

        deform_correct[d_level] += correct
        deform_total[d_level]   += 1
        dirty_correct[e_level]  += correct
        dirty_total[e_level]    += 1

    def _to_acc(c_map, t_map):
        return {
            f"d{lvl}": c_map[lvl] / t_map[lvl]
            for lvl in sorted(t_map) if t_map[lvl] > 0
        }

    def _to_acc_e(c_map, t_map):
        return {
            f"e{lvl}": c_map[lvl] / t_map[lvl]
            for lvl in sorted(t_map) if t_map[lvl] > 0
        }

    d_acc = _to_acc(deform_correct,  deform_total)
    e_acc = _to_acc_e(dirty_correct, dirty_total)
    return d_acc, e_acc


# ── main ──────────────────────────────────────────────────────────────────────
def main() -> None:
    print("=" * 60)
    print("Step 5 — Model Evaluation")
    print("=" * 60)

    print(f"\nLoading model from {CKPT_PATH} …")
    model = load_model()

    print("Running inference on test set …")
    all_preds, all_labels, all_paths = run_inference(model)

    # ── sklearn classification report ─────────────────────────────────────────
    report = classification_report(
        all_labels, all_preds, target_names=CLASS_NAMES, digits=4
    )
    print("\nClassification Report:")
    print(report)

    # ── confusion matrix ──────────────────────────────────────────────────────
    plot_confusion_matrix(all_labels, all_preds)

    # ── WaDaBa group analysis ─────────────────────────────────────────────────
    d_acc, e_acc = wadaba_group_analysis(all_preds, all_labels, all_paths)

    deform_section = ""
    if d_acc:
        deform_section = "\n\nWaDaBa Accuracy by Deformation Level:\n"
        for k, v in sorted(d_acc.items()):
            deform_section += f"  {k}: {v*100:.2f}%\n"
        # Save JSON for step 7
        with open(RESULTS / "deformation_results.json", "w") as fh:
            json.dump(d_acc, fh, indent=2)
        print(deform_section)

    dirty_section = ""
    if e_acc:
        dirty_section = "\n\nWaDaBa Accuracy by Dirtiness Level:\n"
        for k, v in sorted(e_acc.items()):
            dirty_section += f"  {k}: {v*100:.2f}%\n"
        with open(RESULTS / "dirtiness_results.json", "w") as fh:
            json.dump(e_acc, fh, indent=2)
        print(dirty_section)

    # ── write full evaluation report ──────────────────────────────────────────
    report_path = RESULTS / "evaluation_report.txt"
    with open(report_path, "w") as fh:
        fh.write("=" * 60 + "\n")
        fh.write("Plastic Waste Classification — Evaluation Report\n")
        fh.write("=" * 60 + "\n\n")
        fh.write(f"Checkpoint: {CKPT_PATH}\n")
        fh.write(f"Test set:   {TEST_DIR}\n\n")
        fh.write("Classification Report:\n")
        fh.write(report)
        fh.write(deform_section)
        fh.write(dirty_section)

    print(f"\nEvaluation report saved → {report_path}")
    print("=" * 60)


main()


Step 5 — Model Evaluation

Loading model from /kaggle/working/checkpoints/phase3_best.pth …
Running inference on test set …

Classification Report:
              precision    recall  f1-score   support

        HDPE     1.0000    0.9932    0.9966       147
        LDPE     1.0000    1.0000    1.0000        78
       OTHER     0.9916    0.9874    0.9895       238
         PET     1.0000    0.9443    0.9713       341
          PP     0.8793    1.0000    0.9358       102
          PS     0.9054    1.0000    0.9504        67

    accuracy                         0.9764       973
   macro avg     0.9627    0.9875    0.9739       973
weighted avg     0.9788    0.9764    0.9767       973

Confusion matrix saved → /kaggle/working/results/confusion_matrix.png


WaDaBa Accuracy by Deformation Level:
  d0: 91.67%
  d1: 100.00%
  d2: 94.34%
  d3: 88.00%



WaDaBa Accuracy by Dirtiness Level:
  e0: 94.55%
  e1: 88.57%


Evaluation report saved → /kaggle/working/results/evaluation_report.txt


---
## Step 6 — Extras: Grad-CAM / Confidence / ONNX

In [12]:
pip install grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 53.9 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44286 sha256=7a32d82b1f166807a553650a0399771f59b46ec7e43236d963af7686ebd97fee
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam
Note: you may need to restart the kernel to use updated packages.


In [16]:
pip install onnxruntime-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 MB 7.3 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [18]:
pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 9.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 12.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [19]:
"""
Step 6: Extras — Grad-CAM visualisation, confidence threshold rejection, ONNX export.

1. Grad-CAM
   - Target layer: model.features[-1]
   - 10 random test images
   - Side-by-side original | CAM overlay saved to results/gradcam/sample_{N}.png

2. Confidence Threshold (0.70)
   - Images where max softmax probability < 0.70 → flagged as "Unknown — Flag for Review"
   - Per-class flagging stats saved to results/confidence_report.txt

3. ONNX Export
   - Opset 11, input shape (1, 3, 300, 300)
   - Verified with onnxruntime
   - Saved to results/plastic_classifier.onnx

Hardware: GPU required.
Requires: pip install pytorch-grad-cam onnx onnxruntime

Run: python step6_extras.py
"""

import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# pytorch-grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# ONNX
import onnx
import onnxruntime as ort

try:
    from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
    def build_backbone() -> nn.Module:
        return efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
except ImportError:
    import torchvision
    def build_backbone() -> nn.Module:          # type: ignore[misc]
        return torchvision.models.efficientnet_b3(pretrained=False)

# ── GPU guard ─────────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), "CUDA GPU is required."
DEVICE = torch.device("cuda")

# ── paths ─────────────────────────────────────────────────────────────────────
BASE_DIR   = Path("/kaggle/working")
CKPT_PATH  = BASE_DIR / "checkpoints" / "phase3_best.pth"
TEST_DIR   = BASE_DIR / "dataset" / "unified" / "test"
RESULTS    = BASE_DIR / "results"
GRADCAM_DIR = RESULTS / "gradcam"
RESULTS.mkdir(exist_ok=True)
GRADCAM_DIR.mkdir(exist_ok=True)

NUM_CLASSES       = 6
IMG_SIZE          = 300
NUM_WORKERS       = 4
CONFIDENCE_THRESH = 0.70
N_GRADCAM_IMAGES  = 10

CLASS_NAMES = sorted(["HDPE", "LDPE", "OTHER", "PET", "PP", "PS"])

IMG_MEAN = [0.485, 0.456, 0.406]
IMG_STD  = [0.229, 0.224, 0.225]

VAL_TF = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMG_MEAN, std=IMG_STD),
])

# Un-normalise for display (RGB float 0-1)
UNNORM_MEAN = torch.tensor(IMG_MEAN).view(3, 1, 1)
UNNORM_STD  = torch.tensor(IMG_STD).view(3, 1, 1)


def unnormalise(tensor: torch.Tensor) -> np.ndarray:
    """Return (H, W, 3) float32 array in [0, 1]."""
    img = tensor.cpu() * UNNORM_STD + UNNORM_MEAN
    img = img.clamp(0, 1).permute(1, 2, 0).numpy().astype(np.float32)
    return img


# ── model ─────────────────────────────────────────────────────────────────────
def load_model() -> nn.Module:
    model = build_backbone()
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(1536, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.2),
        nn.Linear(512, NUM_CLASSES),
    )
    state = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(state)
    model.to(DEVICE).eval()
    return model


# ── dataset with paths ────────────────────────────────────────────────────────
class ImageFolderWithPaths(ImageFolder):
    def __getitem__(self, index):
        img, label = super().__getitem__(index)
        path = self.samples[index][0]
        return img, label, path


# ═══════════════════════════════════════════════════════════════════════════════
# 1. Grad-CAM
# ═══════════════════════════════════════════════════════════════════════════════
def run_gradcam(model: nn.Module) -> None:
    print("\n[1/3] Generating Grad-CAM visualisations …")

    ds = ImageFolderWithPaths(str(TEST_DIR), transform=VAL_TF)
    indices = random.sample(range(len(ds)), min(N_GRADCAM_IMAGES, len(ds)))

    target_layers = [model.features[-1]]

    # GradCAM context manager handles hooks safely
    with GradCAM(model=model, target_layers=target_layers) as cam:
        for sample_idx, ds_idx in enumerate(indices, start=1):
            img_tensor, label, path = ds[ds_idx]
            input_batch = img_tensor.unsqueeze(0).to(DEVICE)

            # Use the true class as the CAM target
            targets = [ClassifierOutputTarget(label)]
            grayscale_cam = cam(input_tensor=input_batch, targets=targets)
            grayscale_map = grayscale_cam[0]  # (H, W)

            rgb_img = unnormalise(img_tensor)
            overlay = show_cam_on_image(rgb_img, grayscale_map, use_rgb=True)

            fig, axes = plt.subplots(1, 2, figsize=(8, 4))
            axes[0].imshow(rgb_img)
            axes[0].set_title(f"Original\nTrue: {CLASS_NAMES[label]}")
            axes[0].axis("off")

            axes[1].imshow(overlay)
            axes[1].set_title("Grad-CAM overlay")
            axes[1].axis("off")

            fig.suptitle(Path(path).name, fontsize=7)
            fig.tight_layout()
            out = GRADCAM_DIR / f"sample_{sample_idx}.png"
            fig.savefig(out, dpi=120)
            plt.close(fig)
            print(f"  Saved {out}")

    print(f"Grad-CAM images saved to {GRADCAM_DIR}")


# ═══════════════════════════════════════════════════════════════════════════════
# 2. Confidence Threshold
# ═══════════════════════════════════════════════════════════════════════════════
def run_confidence_analysis(model: nn.Module) -> None:
    print(f"\n[2/3] Confidence threshold analysis (threshold={CONFIDENCE_THRESH}) …")

    ds = ImageFolderWithPaths(str(TEST_DIR), transform=VAL_TF)
    loader = DataLoader(
        ds, batch_size=32, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
    )

    flagged_per_class   = {cls: 0 for cls in CLASS_NAMES}
    total_per_class     = {cls: 0 for cls in CLASS_NAMES}
    total_flagged       = 0
    total_images        = 0
    softmax             = nn.Softmax(dim=1)

    with torch.no_grad():
        for imgs, labels, _ in loader:
            imgs   = imgs.to(DEVICE, non_blocking=True)
            logits = model(imgs)
            probs  = softmax(logits)
            max_p, _ = probs.max(dim=1)

            for i, (p, lbl) in enumerate(zip(max_p.cpu().tolist(),
                                              labels.tolist())):
                cls = CLASS_NAMES[lbl]
                total_per_class[cls] += 1
                total_images += 1
                if p < CONFIDENCE_THRESH:
                    flagged_per_class[cls] += 1
                    total_flagged += 1

    report_lines = [
        "=" * 60,
        f"Confidence Threshold Report  (threshold = {CONFIDENCE_THRESH})",
        "=" * 60,
        "",
        f"{'Class':<8} {'Total':>7} {'Flagged':>9} {'Flag%':>8}",
        "-" * 40,
    ]
    for cls in CLASS_NAMES:
        tot = total_per_class[cls]
        flg = flagged_per_class[cls]
        pct = (flg / tot * 100) if tot else 0.0
        report_lines.append(f"{cls:<8} {tot:>7} {flg:>9} {pct:>7.2f}%")

    report_lines += [
        "-" * 40,
        f"{'TOTAL':<8} {total_images:>7} {total_flagged:>9} "
        f"{total_flagged/total_images*100:>7.2f}%",
        "",
        "Images below threshold are flagged as: Unknown — Flag for Review",
    ]

    report_text = "\n".join(report_lines)
    print(report_text)

    out = RESULTS / "confidence_report.txt"
    out.write_text(report_text, encoding="utf-8")
    print(f"\nConfidence report saved → {out}")


# ═══════════════════════════════════════════════════════════════════════════════
# 3. ONNX Export
# ═══════════════════════════════════════════════════════════════════════════════
def export_onnx(model: nn.Module) -> None:
    print("\n[3/3] Exporting model to ONNX …")

    onnx_path = RESULTS / "plastic_classifier.onnx"
    dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

    # Export with dropout disabled (eval mode already set)
    torch.onnx.export(
        model,
        dummy_input,
        str(onnx_path),
        opset_version=11,
        input_names=["image"],
        output_names=["logits"],
        dynamic_axes={"image": {0: "batch_size"}, "logits": {0: "batch_size"}},
        verbose=False,
    )

    # Validate ONNX model structure
    onnx_model = onnx.load(str(onnx_path))
    onnx.checker.check_model(onnx_model)
    print("  ONNX model structure — OK")

    # Verify with onnxruntime
    sess = ort.InferenceSession(str(onnx_path),
                                providers=["CUDAExecutionProvider",
                                           "CPUExecutionProvider"])
    dummy_np = dummy_input.cpu().numpy()
    ort_out   = sess.run(None, {"image": dummy_np})
    torch_out = model(dummy_input).detach().cpu().numpy()

    max_diff = float(np.abs(ort_out[0] - torch_out).max())
    print(f"  onnxruntime vs PyTorch max output diff: {max_diff:.6f}")
    assert max_diff < 1e-2, f"ONNX verification failed: diff={max_diff}"
    print(f"  ONNX verification — OK")
    print(f"  Saved → {onnx_path}")


# ── main ──────────────────────────────────────────────────────────────────────
def main() -> None:
    print("=" * 60)
    print("Step 6 — Extras: Grad-CAM / Confidence / ONNX")
    print("=" * 60)

    model = load_model()

    run_gradcam(model)
    run_confidence_analysis(model)
    export_onnx(model)

    print("\nStep 6 complete.")


main()


Step 6 — Extras: Grad-CAM / Confidence / ONNX

[1/3] Generating Grad-CAM visualisations …
  Saved /kaggle/working/results/gradcam/sample_1.png
  Saved /kaggle/working/results/gradcam/sample_2.png
  Saved /kaggle/working/results/gradcam/sample_3.png
  Saved /kaggle/working/results/gradcam/sample_4.png
  Saved /kaggle/working/results/gradcam/sample_5.png
  Saved /kaggle/working/results/gradcam/sample_6.png
  Saved /kaggle/working/results/gradcam/sample_7.png
  Saved /kaggle/working/results/gradcam/sample_8.png
  Saved /kaggle/working/results/gradcam/sample_9.png
  Saved /kaggle/working/results/gradcam/sample_10.png
Grad-CAM images saved to /kaggle/working/results/gradcam

[2/3] Confidence threshold analysis (threshold=0.7) …
Confidence Threshold Report  (threshold = 0.7)

Class      Total   Flagged    Flag%
----------------------------------------
HDPE         147         1    0.68%
LDPE          78         0    0.00%
OTHER        238        10    4.20%
PET          341        38   11.14

/tmp/ipykernel_55/1357806894.py:239: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0326 01:24:33.542000 55 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0326 01:24:34.296000 55 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligne

Applied 157 of general pattern rewrite rules.
  ONNX model structure — OK
  onnxruntime vs PyTorch max output diff: 0.000001
  ONNX verification — OK
  Saved → /kaggle/working/results/plastic_classifier.onnx

Step 6 complete.


---
## Step 7 — Visualise Results

In [20]:
"""
Step 7: Visualisation plots.

Produces three PNG files in results/:

1. training_curves.png
   - 2-subplot figure (train/val loss  |  train/val accuracy)
   - One line per phase (phases 1, 2, 3) on a continuous x-axis
   - Vertical dashed lines mark phase boundaries

2. class_distribution.png
   - Horizontal bar chart of per-class image counts in dataset/unified/train/
   - Two stacked bars: TIP images vs. WaDaBa images

3. deformation_accuracy.png
   - Bar chart of WaDaBa test accuracy per deformation level (d0–d3)
   - Data read from results/deformation_results.json  (produced by step 5)

Run: python step7_visualize.py
"""

import json
import os
from pathlib import Path
from collections import defaultdict

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

# ── paths ─────────────────────────────────────────────────────────────────────
BASE_DIR   = Path(__file__).parent
LOG_DIR    = BASE_DIR / "logs"
RESULTS    = BASE_DIR / "results"
UNIFIED_TRAIN = BASE_DIR / "dataset" / "unified" / "train"
DEFORM_JSON   = RESULTS / "deformation_results.json"
RESULTS.mkdir(exist_ok=True)

CLASS_NAMES = sorted(["HDPE", "LDPE", "OTHER", "PET", "PP", "PS"])
PHASE_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c"]   # blue, orange, green


# ═══════════════════════════════════════════════════════════════════════════════
# 1. Training curves
# ═══════════════════════════════════════════════════════════════════════════════
def plot_training_curves() -> None:
    out = RESULTS / "training_curves.png"

    phases: list[pd.DataFrame] = []
    for p in [1, 2, 3]:
        log_path = LOG_DIR / f"phase{p}_log.csv"
        if not log_path.exists():
            print(f"  [WARN] {log_path} not found — training_curves.png may be incomplete")
            phases.append(pd.DataFrame())
        else:
            phases.append(pd.read_csv(log_path))

    if all(df.empty for df in phases):
        print("  [WARN] No phase logs found — skipping training_curves.png")
        return

    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Training Curves", fontsize=14, fontweight="bold")

    global_offset = 0
    phase_boundaries: list[int] = []

    for p_idx, (df, color) in enumerate(zip(phases, PHASE_COLORS), start=1):
        if df.empty:
            continue

        x = np.arange(global_offset + 1, global_offset + len(df) + 1)

        ax_loss.plot(x, df["train_loss"], color=color, linestyle="-",
                     label=f"Phase {p_idx} train")
        ax_loss.plot(x, df["val_loss"],   color=color, linestyle="--",
                     label=f"Phase {p_idx} val")

        ax_acc.plot(x, df["train_acc"], color=color, linestyle="-",
                    label=f"Phase {p_idx} train")
        ax_acc.plot(x, df["val_acc"],   color=color, linestyle="--",
                    label=f"Phase {p_idx} val")

        global_offset += len(df)
        phase_boundaries.append(global_offset)

    # Vertical phase separators (skip the last one)
    for boundary in phase_boundaries[:-1]:
        ax_loss.axvline(x=boundary + 0.5, color="gray", linestyle=":", alpha=0.6)
        ax_acc.axvline( x=boundary + 0.5, color="gray", linestyle=":", alpha=0.6)

    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel("Loss")
    ax_loss.set_title("Train / Val Loss")
    ax_loss.legend(fontsize=8)
    ax_loss.grid(True, alpha=0.3)

    ax_acc.set_xlabel("Epoch")
    ax_acc.set_ylabel("Accuracy")
    ax_acc.set_title("Train / Val Accuracy")
    ax_acc.legend(fontsize=8)
    ax_acc.grid(True, alpha=0.3)
    ax_acc.set_ylim(0.0, 1.05)

    fig.tight_layout()
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"Training curves saved → {out}")


# ═══════════════════════════════════════════════════════════════════════════════
# 2. Class distribution (stacked TIP / WaDaBa)
# ═══════════════════════════════════════════════════════════════════════════════
def plot_class_distribution() -> None:
    out = RESULTS / "class_distribution.png"

    if not UNIFIED_TRAIN.is_dir():
        print(f"  [WARN] {UNIFIED_TRAIN} not found — skipping class_distribution.png")
        return

    tip_counts    = defaultdict(int)
    wadaba_counts = defaultdict(int)

    for cls in CLASS_NAMES:
        cls_dir = UNIFIED_TRAIN / cls
        if not cls_dir.is_dir():
            continue
        for img_path in cls_dir.glob("*.jpg"):
            if img_path.name.startswith("tip_"):
                tip_counts[cls] += 1
            elif img_path.name.startswith("wadaba_"):
                wadaba_counts[cls] += 1

    tip_vals    = [tip_counts[c]    for c in CLASS_NAMES]
    wadaba_vals = [wadaba_counts[c] for c in CLASS_NAMES]
    totals      = [t + w for t, w in zip(tip_vals, wadaba_vals)]

    y_pos = np.arange(len(CLASS_NAMES))
    fig, ax = plt.subplots(figsize=(10, 6))

    bars_tip    = ax.barh(y_pos, tip_vals, color="#4C72B0", label="TIP")
    bars_wadaba = ax.barh(y_pos, wadaba_vals, left=tip_vals,
                          color="#DD8452", label="WaDaBa")

    # Annotate total counts
    for i, total in enumerate(totals):
        ax.text(total + 5, i, str(total), va="center", fontsize=9)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Number of images")
    ax.set_title("Training Set Class Distribution (dataset/unified/train/)")
    ax.legend()
    ax.grid(axis="x", alpha=0.3)

    fig.tight_layout()
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"Class distribution saved → {out}")


# ═══════════════════════════════════════════════════════════════════════════════
# 3. Deformation accuracy
# ═══════════════════════════════════════════════════════════════════════════════
def plot_deformation_accuracy() -> None:
    out = RESULTS / "deformation_accuracy.png"

    if not DEFORM_JSON.exists():
        print(f"  [WARN] {DEFORM_JSON} not found — skipping deformation_accuracy.png")
        return

    with open(DEFORM_JSON) as fh:
        d_acc: dict[str, float] = json.load(fh)

    labels = sorted(d_acc.keys())                       # d0, d1, d2, d3
    values = [d_acc[k] * 100.0 for k in labels]
    level_names = {
        "d0": "d0 — None",
        "d1": "d1 — Low",
        "d2": "d2 — Medium",
        "d3": "d3 — High",
    }
    x_labels = [level_names.get(l, l) for l in labels]

    fig, ax = plt.subplots(figsize=(8, 5))
    colors  = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]
    bars = ax.bar(x_labels, values, color=colors[: len(labels)], width=0.5)

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2.0,
                bar.get_height() + 0.5,
                f"{val:.1f}%", ha="center", va="bottom", fontsize=10)

    ax.set_xlabel("Deformation Level")
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("WaDaBa Test Accuracy by Deformation Level")
    ax.set_ylim(0, 110)
    ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig(out, dpi=150)
    plt.close(fig)
    print(f"Deformation accuracy chart saved → {out}")


# ── main ──────────────────────────────────────────────────────────────────────
def main() -> None:
    print("=" * 60)
    print("Step 7 — Visualisation")
    print("=" * 60)

    plot_training_curves()
    plot_class_distribution()
    plot_deformation_accuracy()

    print("\nAll plots saved to", RESULTS)
    print("=" * 60)


main()


Step 7 — Visualisation
Training curves saved → /kaggle/working/results/training_curves.png
Class distribution saved → /kaggle/working/results/class_distribution.png
Deformation accuracy chart saved → /kaggle/working/results/deformation_accuracy.png

All plots saved to /kaggle/working/results


---
## Conveyor Belt Real-World Evaluation

In [22]:
"""
Conveyor Belt Real-World Evaluation
=====================================
Evaluates checkpoints/phase3_best.pth on the conveyor belt dataset.
This dataset is used ONLY for testing — never for training.

Dataset layout:
  conveyor/img/           ← 95 labeled JPEG images (640×640)
  conveyor/labels/        ← Pascal VOC XML files with bounding boxes
  conveyor/additional_images/ ← unlabeled, ignored

Class mapping:
  mix PP    → PP
  mix HD    → HDPE
  mix PET   → PET
  mix rigid → OTHER

Outputs (all in results/):
  conveyor_eval_report.txt
  conveyor_confusion_matrix.png
  conveyor_sample_predictions.png
  conveyor_confidence_distribution.png
  conveyor_annotated_images/  ← per-image bounding-box overlay

Run: python step_conveyor_eval.py
"""

import os
import re
import random
import warnings
import xml.etree.ElementTree as ET
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageDraw, ImageFont
from torchvision import transforms
from tqdm import tqdm

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# ── GPU guard ──────────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), (
    "CUDA GPU is required for conveyor evaluation."
)
DEVICE = torch.device("cuda")
print(f"Device: {torch.cuda.get_device_name(0)}")

# ── paths ──────────────────────────────────────────────────────────────────────

# Kaggle working directory (for outputs)
BASE_DIR = Path("/kaggle/working")

# Model checkpoint (⚠️ update if needed)
CKPT_PATH = BASE_DIR / "checkpoints" / "phase3_best.pth"

# Dataset paths (your provided paths)
CONVEYOR_IMG = Path("/kaggle/input/datasets/durvibangera/dataset/conveyor/conveyor/img")
CONVEYOR_LABELS = Path("/kaggle/input/datasets/durvibangera/dataset/conveyor/conveyor/labels")

# Output directories
RESULTS = BASE_DIR / "results"
ANNOTATED_DIR = RESULTS / "conveyor_annotated_images"

# Report file
EVAL_REPORT_TXT = RESULTS / "evaluation_report.txt"

# Create folders (safe in Kaggle)
RESULTS.mkdir(parents=True, exist_ok=True)
ANNOTATED_DIR.mkdir(parents=True, exist_ok=True)

# ── constants ──────────────────────────────────────────────────────────────────
NUM_CLASSES       = 6
IMG_SIZE          = 300
CONF_THRESHOLD    = 0.70
CLASS_NAMES       = sorted(["HDPE", "LDPE", "OTHER", "PET", "PP", "PS"])
CLASS_TO_IDX      = {c: i for i, c in enumerate(CLASS_NAMES)}

CONVEYOR_CLASS_MAP = {
    "mix pp":    "PP",
    "mix hd":    "HDPE",
    "mix hdpe":  "HDPE",
    "mix pet":   "PET",
    "mix rigid": "OTHER",
}

# ── inference transform (no augmentation) ─────────────────────────────────────
INFER_TF = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ── Test-Time Augmentation (TTA) transforms ────────────────────────────────────
# 8 deterministic views per crop: baseline, flips, rotations, brightness shifts.
# Averaging softmax over all views reduces sensitivity to pose/lighting variation
# — the primary source of domain shift on real conveyor footage.
_norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
TTA_TRANSFORMS = [
    # 1. baseline
    transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),
                        transforms.ToTensor(), _norm]),
    # 2. horizontal flip
    transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),
                        transforms.RandomHorizontalFlip(p=1.0),
                        transforms.ToTensor(), _norm]),
    # 3. vertical flip
    transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),
                        transforms.RandomVerticalFlip(p=1.0),
                        transforms.ToTensor(), _norm]),
    # 4. +15° rotation
    transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),
                        transforms.RandomRotation((15, 15)),
                        transforms.ToTensor(), _norm]),
    # 5. −15° rotation
    transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),
                        transforms.RandomRotation((-15, -15)),
                        transforms.ToTensor(), _norm]),
    # 6. h-flip + 10° rotation
    transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),
                        transforms.RandomHorizontalFlip(p=1.0),
                        transforms.RandomRotation((10, 10)),
                        transforms.ToTensor(), _norm]),
    # 7. 20% brighter
    transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),
                        transforms.ColorJitter(brightness=(1.2, 1.2)),
                        transforms.ToTensor(), _norm]),
    # 8. 25% darker
    transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)),
                        transforms.ColorJitter(brightness=(0.75, 0.75)),
                        transforms.ToTensor(), _norm]),
]


# ── model ─────────────────────────────────────────────────────────────────────
def build_model() -> nn.Module:
    try:
        from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
        model = efficientnet_b3(weights=None)
    except ImportError:
        import torchvision
        model = torchvision.models.efficientnet_b3(pretrained=False)

    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(1536, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.2),
        nn.Linear(512, NUM_CLASSES),
    )
    state = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(state)
    model.to(DEVICE).eval()
    return model


# ── XML parsing ───────────────────────────────────────────────────────────────
def parse_xml(xml_path: Path) -> list[dict]:
    """
    Parse a Pascal VOC XML file and return a list of annotation dicts:
      [{"class": "PP", "box": (xmin, ymin, xmax, ymax)}, ...]
    Returns an empty list on parse failure (logs a warning).
    """
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
    except ET.ParseError as exc:
        warnings.warn(f"Malformed XML skipped: {xml_path}  ({exc})")
        return []

    objects = []
    for obj in root.findall("object"):
        raw_name = (obj.findtext("name") or "").strip().lower()
        target_class = CONVEYOR_CLASS_MAP.get(raw_name)
        if target_class is None:
            warnings.warn(f"Unknown class '{raw_name}' in {xml_path}, skipping object.")
            continue

        bndbox = obj.find("bndbox")
        if bndbox is None:
            continue
        try:
            xmin = int(float(bndbox.findtext("xmin")))
            ymin = int(float(bndbox.findtext("ymin")))
            xmax = int(float(bndbox.findtext("xmax")))
            ymax = int(float(bndbox.findtext("ymax")))
        except (TypeError, ValueError):
            warnings.warn(f"Bad bndbox values in {xml_path}, skipping object.")
            continue

        objects.append({"class": target_class, "box": (xmin, ymin, xmax, ymax)})
    return objects


def find_xml_for_image(img_path: Path) -> Path | None:
    """
    Look for a matching XML in CONVEYOR_LABELS.
    Tries exact stem match first, then a case-insensitive search.
    """
    stem = img_path.stem
    # exact match
    candidate = CONVEYOR_LABELS / (stem + ".xml")
    if candidate.exists():
        return candidate
    # case-insensitive fallback
    stem_lower = stem.lower()
    for xml_file in CONVEYOR_LABELS.glob("*.xml"):
        if xml_file.stem.lower() == stem_lower:
            return xml_file
    return None


# ── inference on a single crop ────────────────────────────────────────────────
@torch.no_grad()
def predict_crop(model: nn.Module, crop: Image.Image) -> tuple[str, float]:
    """
    Returns (predicted_class_or_Unknown, confidence) using Test-Time Augmentation.

    Runs each crop through 8 deterministic augmented views (flips, rotations,
    brightness shifts) and averages the softmax probabilities before taking
    argmax.  This reduces sensitivity to pose and lighting variation — the
    dominant source of domain shift on real conveyor imagery.
    """
    if crop.width < 2 or crop.height < 2:
        return "Unknown", 0.0

    crop_rgb  = crop.convert("RGB")
    probs_sum = torch.zeros(NUM_CLASSES, device=DEVICE)

    for tf in TTA_TRANSFORMS:
        tensor     = tf(crop_rgb).unsqueeze(0).to(DEVICE)
        probs_sum += torch.softmax(model(tensor), dim=1).squeeze(0)

    avg_probs = probs_sum / len(TTA_TRANSFORMS)
    conf      = avg_probs.max().item()
    pred_idx  = avg_probs.argmax().item()

    if conf < CONF_THRESHOLD:
        return "Unknown", conf
    return CLASS_NAMES[pred_idx], conf


# ── draw annotated image ──────────────────────────────────────────────────────
def save_annotated_image(
    img: Image.Image,
    annotations: list[dict],
    predictions: list[tuple[str, float]],
    out_path: Path,
) -> None:
    """Draw bounding boxes on img and save. Green = correct, Red = wrong."""
    draw = ImageDraw.Draw(img.copy())
    result_img = img.copy()
    draw = ImageDraw.Draw(result_img)

    try:
        font = ImageFont.truetype("arial.ttf", size=12)
    except OSError:
        font = ImageFont.load_default()

    for ann, (pred_class, conf) in zip(annotations, predictions):
        gt   = ann["class"]
        box  = ann["box"]
        correct = (pred_class == gt)
        color = "#00cc44" if correct else "#ff3333"   # green / red

        draw.rectangle(box, outline=color, width=2)
        label = f"GT:{gt}|P:{pred_class}|{conf*100:.0f}%"

        # draw label background
        bbox_text = draw.textbbox((box[0], box[1] - 15), label, font=font)
        draw.rectangle(bbox_text, fill=color)
        draw.text((box[0], box[1] - 15), label, fill="white", font=font)

    result_img.save(out_path)


# ── load clean-test accuracy from existing report ────────────────────────────
def load_clean_accuracy() -> float | None:
    if not EVAL_REPORT_TXT.exists():
        return None
    text = EVAL_REPORT_TXT.read_text(encoding="utf-8", errors="ignore")
    # look for "accuracy   …   0.XXXX" line in sklearn report
    m = re.search(r"accuracy\s+(\d+\.\d+)\s+\d+", text)
    if m:
        return float(m.group(1)) * 100.0
    return None


# ── confidence distribution from clean test ──────────────────────────────────
def collect_clean_confidences(model: nn.Module) -> list[float]:
    """Run model on unified/test/ and collect max-softmax confidence per sample."""
    test_dir = BASE_DIR / "dataset" / "unified" / "test"
    if not test_dir.exists():
        return []

    from torchvision.datasets import ImageFolder
    from torch.utils.data import DataLoader

    ds     = ImageFolder(str(test_dir), transform=INFER_TF)
    loader = DataLoader(ds, batch_size=64, shuffle=False,
                        num_workers=0, pin_memory=True)
    confs  = []
    with torch.no_grad():
        for imgs, _ in tqdm(loader, desc="Clean-test confidences", leave=False):
            imgs  = imgs.to(DEVICE, non_blocking=True)
            probs = torch.softmax(model(imgs), dim=1)
            confs.extend(probs.max(dim=1).values.cpu().tolist())
    return confs


# ── confusion matrix plot ─────────────────────────────────────────────────────
def plot_confusion_matrix(
    y_true_names: list[str],
    y_pred_names: list[str],
    labels: list[str],
    title: str,
    out_path: Path,
) -> None:
    label_set = sorted(set(y_true_names) | set(y_pred_names) - {"Unknown"})
    # keep only classes present in GT
    gt_classes = sorted(set(y_true_names))
    cm = confusion_matrix(y_true_names, y_pred_names, labels=gt_classes)
    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    n = len(gt_classes)
    fig, ax = plt.subplots(figsize=(max(6, n), max(5, n - 1)))
    img = ax.imshow(cm_norm, interpolation="nearest", cmap="Blues",
                    vmin=0.0, vmax=1.0)
    plt.colorbar(img, ax=ax)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(gt_classes, rotation=45, ha="right")
    ax.set_yticklabels(gt_classes)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)

    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{cm_norm[i, j]:.2f}",
                    ha="center", va="center",
                    color="white" if cm_norm[i, j] > 0.5 else "black",
                    fontsize=9)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f"Confusion matrix saved → {out_path}")


# ── sample predictions grid ──────────────────────────────────────────────────
def plot_sample_predictions(
    crop_records: list[dict],
    out_path: Path,
) -> None:
    """Pick 3 crops per present class (up to 12 total) and show a 3×4 grid."""
    by_class: dict[str, list[dict]] = defaultdict(list)
    for rec in crop_records:
        by_class[rec["gt"]].append(rec)

    target_classes = ["PP", "HDPE", "PET", "OTHER"]
    samples: list[dict] = []
    for cls in target_classes:
        pool = by_class.get(cls, [])
        random.shuffle(pool)
        samples.extend(pool[:3])

    if not samples:
        return

    cols, rows = 4, 3
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes_flat = axes.flatten()

    for i, ax in enumerate(axes_flat):
        ax.axis("off")
        if i >= len(samples):
            continue
        rec  = samples[i]
        crop = rec["crop_pil"]
        gt   = rec["gt"]
        pred = rec["pred"]
        conf = rec["conf"]

        ax.imshow(crop)
        color = "green" if pred == gt else "red"
        ax.set_title(f"GT: {gt} | Pred: {pred} | {conf*100:.1f}%",
                     color=color, fontsize=8, fontweight="bold")

    fig.suptitle("Conveyor Belt Sample Predictions (3 per class)", fontsize=11)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f"Sample predictions saved → {out_path}")


# ── confidence distribution plot ─────────────────────────────────────────────
def plot_confidence_distribution(
    clean_confs: list[float],
    conveyor_confs: list[float],
    out_path: Path,
) -> None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
    bins = np.linspace(0, 1, 21)

    ax1.hist(clean_confs, bins=bins, color="#4c78a8", edgecolor="white", linewidth=0.4)
    ax1.axvline(CONF_THRESHOLD, color="red", linestyle="--", linewidth=1.2,
                label=f"Threshold {CONF_THRESHOLD}")
    ax1.set_title("Clean Test Set")
    ax1.set_xlabel("Max Softmax Confidence")
    ax1.set_ylabel("Count")
    ax1.legend(fontsize=8)

    ax2.hist(conveyor_confs, bins=bins, color="#f58518", edgecolor="white", linewidth=0.4)
    ax2.axvline(CONF_THRESHOLD, color="red", linestyle="--", linewidth=1.2,
                label=f"Threshold {CONF_THRESHOLD}")
    ax2.set_title("Conveyor Belt (Real-World)")
    ax2.set_xlabel("Max Softmax Confidence")
    ax2.set_ylabel("Count")
    ax2.legend(fontsize=8)

    fig.suptitle("Confidence Distribution: Clean vs Real-World", fontsize=12)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f"Confidence distribution saved → {out_path}")


# ── main ──────────────────────────────────────────────────────────────────────
def main() -> None:
    random.seed(42)
    model = build_model()
    print(f"Model loaded from {CKPT_PATH}")

    # Collect all labeled images
    img_files = sorted(CONVEYOR_IMG.glob("*.jp*g")) + sorted(CONVEYOR_IMG.glob("*.JP*G"))
    # deduplicate by stem (case-insensitive)
    seen_stems: set[str] = set()
    unique_imgs: list[Path] = []
    for p in img_files:
        if p.stem.lower() not in seen_stems:
            seen_stems.add(p.stem.lower())
            unique_imgs.append(p)
    img_files = unique_imgs

    # Per-class accumulators
    per_class_gt:   dict[str, int]   = defaultdict(int)
    per_class_ok:   dict[str, int]   = defaultdict(int)
    per_class_conf: dict[str, list]  = defaultdict(list)

    correct_confs: list[float]  = []
    wrong_confs:   list[float]  = []
    unknown_count: int          = 0
    conveyor_confs: list[float] = []

    all_gt_names:   list[str] = []
    all_pred_names: list[str] = []

    crop_records: list[dict] = []   # for sample-prediction grid

    images_evaluated = 0
    total_crops      = 0

    for img_path in tqdm(img_files, desc="Evaluating images"):
        xml_path = find_xml_for_image(img_path)
        if xml_path is None:
            warnings.warn(f"No XML found for {img_path.name}, skipping.")
            continue

        annotations = parse_xml(xml_path)
        if not annotations:
            continue

        try:
            pil_img = Image.open(img_path).convert("RGB")
        except Exception as exc:
            warnings.warn(f"Cannot open {img_path.name}: {exc}, skipping.")
            continue

        images_evaluated += 1
        predictions: list[tuple[str, float]] = []

        for ann in annotations:
            gt   = ann["class"]
            box  = ann["box"]
            xmin, ymin, xmax, ymax = box

            # Clamp box to image dimensions
            w, h = pil_img.size
            xmin = max(0, min(xmin, w - 1))
            xmax = max(xmin + 1, min(xmax, w))
            ymin = max(0, min(ymin, h - 1))
            ymax = max(ymin + 1, min(ymax, h))

            crop = pil_img.crop((xmin, ymin, xmax, ymax))
            pred_class, conf = predict_crop(model, crop)

            predictions.append((pred_class, conf))
            conveyor_confs.append(conf)
            total_crops += 1

            per_class_gt[gt] += 1
            per_class_conf[gt].append(conf)

            if pred_class == "Unknown":
                unknown_count += 1
                all_gt_names.append(gt)
                all_pred_names.append("Unknown")
                wrong_confs.append(conf)
            else:
                all_gt_names.append(gt)
                all_pred_names.append(pred_class)
                if pred_class == gt:
                    per_class_ok[gt] += 1
                    correct_confs.append(conf)
                else:
                    wrong_confs.append(conf)

            crop_records.append({
                "image":    img_path.name,
                "gt":       gt,
                "pred":     pred_class,
                "conf":     conf,
                "crop_pil": crop,
            })

        # Save annotated image
        ann_out = ANNOTATED_DIR / img_path.name
        save_annotated_image(pil_img, annotations, predictions, ann_out)

    # ── metrics ───────────────────────────────────────────────────────────────
    overall_correct = sum(
        1 for g, p in zip(all_gt_names, all_pred_names) if g == p
    )
    overall_acc = (overall_correct / total_crops * 100) if total_crops else 0.0
    avg_conf_correct = np.mean(correct_confs) * 100 if correct_confs else 0.0
    avg_conf_wrong   = np.mean(wrong_confs)   * 100 if wrong_confs   else 0.0
    unknown_pct      = (unknown_count / total_crops * 100) if total_crops else 0.0

    # ── comparison vs clean test set ─────────────────────────────────────────
    clean_acc = load_clean_accuracy()
    acc_drop  = (clean_acc - overall_acc) if clean_acc is not None else None

    clean_confs = collect_clean_confidences(model)
    avg_clean_conf    = np.mean(clean_confs)    * 100 if clean_confs    else 0.0
    avg_conveyor_conf = np.mean(conveyor_confs) * 100 if conveyor_confs else 0.0
    conf_drop = avg_clean_conf - avg_conveyor_conf

    # ── write report ──────────────────────────────────────────────────────────
    report_lines: list[str] = []
    w = report_lines.append

    w("Conveyor Belt Real-World Evaluation")
    w("=====================================")
    w(f"Total images evaluated     : {images_evaluated}")
    w(f"Total bounding box crops   : {total_crops}")
    w(f"Overall crop accuracy      : {overall_acc:.1f}%")
    w("")
    w("Per-class Results:")
    header = f"  {'Class':<10} {'GT crops':>10} {'Correct':>9} {'Accuracy':>10} {'Avg Confidence':>16}"
    w(header)
    w("  " + "-" * (len(header) - 2))

    conveyor_classes = ["PP", "HDPE", "PET", "OTHER"]
    for cls in conveyor_classes:
        gt_n  = per_class_gt.get(cls, 0)
        ok_n  = per_class_ok.get(cls, 0)
        acc_c = (ok_n / gt_n * 100) if gt_n else 0.0
        cf    = (np.mean(per_class_conf[cls]) * 100) if per_class_conf[cls] else 0.0
        w(f"  {cls:<10} {gt_n:>10} {ok_n:>9} {acc_c:>9.1f}% {cf:>14.1f}%")

    w("")
    w("Confidence Analysis:")
    w(f"  Average confidence (correct predictions)   : {avg_conf_correct:.1f}%")
    w(f"  Average confidence (wrong predictions)     : {avg_conf_wrong:.1f}%")
    w(f"  Crops flagged as Unknown (conf < {CONF_THRESHOLD:.2f})     : {unknown_count} ({unknown_pct:.1f}%)")

    w("")
    w("Comparison vs Clean Test Set:")
    if clean_acc is not None:
        w(f"  Clean test accuracy    : {clean_acc:.1f}%   (from results/evaluation_report.txt)")
    else:
        w("  Clean test accuracy    : N/A  (results/evaluation_report.txt not found)")
    w(f"  Conveyor accuracy      : {overall_acc:.1f}%")
    if acc_drop is not None:
        w(f"  Accuracy drop          : {acc_drop:.1f}%   (expected — real world is harder)")
    else:
        w("  Accuracy drop          : N/A")
    w(f"  Confidence drop        : {conf_drop:.1f}%")

    report_text = "\n".join(report_lines)
    report_path = RESULTS / "conveyor_eval_report.txt"
    report_path.write_text(report_text, encoding="utf-8")
    print(f"\nReport saved → {report_path}")
    print()
    print(report_text)

    # ── visualizations ────────────────────────────────────────────────────────
    # 1. Confusion matrix (exclude Unknown from GT axis)
    gt_for_cm   = [g for g, p in zip(all_gt_names, all_pred_names) if g != "Unknown"]
    pred_for_cm = [p for g, p in zip(all_gt_names, all_pred_names) if g != "Unknown"]
    if gt_for_cm:
        plot_confusion_matrix(
            gt_for_cm, pred_for_cm,
            labels=conveyor_classes,
            title="Conveyor Belt — Confusion Matrix",
            out_path=RESULTS / "conveyor_confusion_matrix.png",
        )

    # 2. Sample predictions grid
    plot_sample_predictions(crop_records, RESULTS / "conveyor_sample_predictions.png")

    # 3. Confidence distribution
    plot_confidence_distribution(
        clean_confs, conveyor_confs,
        RESULTS / "conveyor_confidence_distribution.png",
    )

    # ── one-line summary ──────────────────────────────────────────────────────
    if clean_acc is not None and acc_drop is not None:
        print(
            f"\nConveyor Real-World Test: {overall_acc:.1f}% crop accuracy on "
            f"{total_crops} crops from {images_evaluated} industrial images\n"
            f"(vs {clean_acc:.1f}% on clean test set — {acc_drop:.1f}% domain gap)"
        )
    else:
        print(
            f"\nConveyor Real-World Test: {overall_acc:.1f}% crop accuracy on "
            f"{total_crops} crops from {images_evaluated} industrial images"
        )


main()


Device: Tesla T4
Model loaded from /kaggle/working/checkpoints/phase3_best.pth


Evaluating images:  34%|███▎      | 32/95 [00:23<00:45,  1.37it/s]/tmp/ipykernel_55/2770107948.py:473: UserWarning: No XML found for IN-SURAT-EV-01-03-2025-VU0001_20250425_160651_Color_Mix_HDPE_rigid.jpeg, skipping.
  warnings.warn(f"No XML found for {img_path.name}, skipping.")
Evaluating images:  75%|███████▍  | 71/95 [00:44<00:15,  1.50it/s]/tmp/ipykernel_55/2770107948.py:473: UserWarning: No XML found for IN-UMBERGAUM-LP-24-09-2025-VU0001_20260127_110138_ - Copy.jpeg, skipping.
  warnings.warn(f"No XML found for {img_path.name}, skipping.")
Evaluating images: 100%|██████████| 95/95 [01:01<00:00,  1.55it/s]



Report saved → /kaggle/working/results/conveyor_eval_report.txt

Conveyor Belt Real-World Evaluation
Total images evaluated     : 93
Total bounding box crops   : 462
Overall crop accuracy      : 33.1%

Per-class Results:
  Class        GT crops   Correct   Accuracy   Avg Confidence
  -----------------------------------------------------------
  PP                136       107      78.7%           78.5%
  HDPE               38        35      92.1%           93.2%
  PET               187         1       0.5%           49.1%
  OTHER             101        10       9.9%           55.0%

Confidence Analysis:
  Average confidence (correct predictions)   : 85.7%
  Average confidence (wrong predictions)     : 51.3%
  Crops flagged as Unknown (conf < 0.70)     : 297 (64.3%)

Comparison vs Clean Test Set:
  Clean test accuracy    : 97.6%   (from results/evaluation_report.txt)
  Conveyor accuracy      : 33.1%
  Accuracy drop          : 64.5%   (expected — real world is harder)
  Confidence drop 

In [23]:
!ls /kaggle/working/checkpoints

phase1_best.pth  phase2_best.pth  phase3_best.pth


In [24]:
!zip -r model_backup.zip /kaggle/working/checkpoints

  adding: kaggle/working/checkpoints/ (stored 0%)
  adding: kaggle/working/checkpoints/phase2_best.pth (deflated 8%)
  adding: kaggle/working/checkpoints/phase1_best.pth (deflated 8%)
  adding: kaggle/working/checkpoints/phase3_best.pth (deflated 7%)


---
## Step 8 — Hierarchical Grade Classification (CLIP Zero-Shot)

In [34]:
!pip uninstall -y clip
!pip install git+https://github.com/openai/CLIP.git

Found existing installation: clip 1.0
Uninstalling clip-1.0:
  Successfully uninstalled clip-1.0
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-_gjhumht
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-_gjhumht
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=a262d07718007445dc5d5c3f3eeb1d2ef59ea1da2c1069d350d1c76ca1bccb6d
  Stored in directory: /tmp/pip-ephem-wheel-cache-ye_skgkv/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [35]:
"""
Step 8: Hierarchical Grade Classification using CLIP zero-shot inference.

After type classification by EfficientNet-B3, this step assesses the
recyclability grade of each plastic item:

  Grade A → Clean, intact, high confidence → Send to recycling stream
  Grade B → Moderate condition             → Pre-process before recycling  
  Grade C → Poor condition, low confidence → Reject or manual review

Uses OpenAI CLIP (ViT-B/32) with natural language grade descriptions.
No additional training required — zero-shot inference only.

Outputs:
  results/grade_sample_predictions.png  — grid of sample grade predictions
  results/grade_report.txt              — per-grade distribution and confidence
"""

# ── Imports ──────────────────────────────────────────────────────────────
import torch
import clip
import numpy as np
from PIL import Image
from pathlib import Path
import torchvision.transforms as T
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm import tqdm
import random

# pip install git+https://github.com/openai/CLIP.git

# ── Device ───────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert torch.cuda.is_available(), "GPU required"
print(f"Device: {device} ({torch.cuda.get_device_name(0)})")

# ── Load CLIP ─────────────────────────────────────────────────────────────
print("Loading CLIP ViT-B/32...")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()
print("CLIP loaded")

# ── Grade Descriptions ────────────────────────────────────────────────────
# Natural language descriptions for each grade
# Written to match what CLIP was trained on (descriptive, visual language)
grade_descriptions = [
    "a clean intact plastic item in good condition suitable for recycling",
    "a slightly dirty or mildly damaged plastic item that needs processing before recycling",
    "a heavily contaminated crushed or severely damaged plastic item not suitable for recycling"
]
grade_labels = ['A', 'B', 'C']
grade_actions = {
    'A': 'Send directly to recycling stream',
    'B': 'Pre-process before recycling',
    'C': 'Reject — do not recycle'
}
grade_colors = {'A': 'green', 'B': 'orange', 'C': 'red'}

# Encode text descriptions once (reuse for all images)
with torch.no_grad():
    text_tokens = clip.tokenize(grade_descriptions).to(device)
    text_features = clip_model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

print("Grade text features encoded")

# ── Grade Classification Function ─────────────────────────────────────────
def classify_grade(pil_image):
    """
    Given a PIL image, return grade (A/B/C), confidence, and action.
    """
    image_input = clip_preprocess(pil_image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        image_features = clip_model.encode_image(image_input)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        
        similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
        confidence, pred_idx = similarity[0].max(dim=-1)
    
    grade = grade_labels[pred_idx.item()]
    conf = confidence.item()
    
    return {
        'grade': grade,
        'confidence': conf,
        'action': grade_actions[grade],
        'all_scores': {
            grade_labels[i]: similarity[0][i].item() 
            for i in range(3)
        }
    }

# ── Full Hierarchical Inference Function ──────────────────────────────────
# Load EfficientNet from phase3 checkpoint
import torchvision.models as models
import torch.nn as nn

NUM_CLASSES = 6
CLASS_NAMES = ['HDPE', 'LDPE', 'OTHER', 'PET', 'PP', 'PS']  # ImageFolder alphabetical order
CONFIDENCE_THRESHOLD = 0.70

def load_efficientnet(checkpoint_path):
    model = models.efficientnet_b3(pretrained=False)
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(1536, 512),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(512, NUM_CLASSES)
    )
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint.get('model_state_dict', checkpoint)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model

efficientnet = load_efficientnet("checkpoints/phase3_best.pth")

# Inference transform for EfficientNet (no augmentation)
infer_transform = T.Compose([
    T.Resize((300, 300)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def classify_full(pil_image):
    """
    Full hierarchical classification:
    Stage 1 → EfficientNet: plastic type
    Stage 2 → CLIP: recyclability grade
    """
    # Stage 1 — Type
    tensor = infer_transform(pil_image).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = efficientnet(tensor)
        probs = torch.softmax(logits, dim=1)
        type_conf, type_pred = probs.max(dim=1)
    
    type_conf = type_conf.item()
    plastic_type = CLASS_NAMES[type_pred.item()]
    
    if type_conf < CONFIDENCE_THRESHOLD:
        return {
            'type': 'Unknown',
            'type_confidence': type_conf,
            'grade': 'Unknown',
            'grade_confidence': 0.0,
            'action': 'Flag for human review'
        }
    
    # Stage 2 — Grade
    grade_result = classify_grade(pil_image)
    
    return {
        'type': plastic_type,
        'type_confidence': type_conf,
        'grade': grade_result['grade'],
        'grade_confidence': grade_result['confidence'],
        'action': grade_result['action'],
        'grade_scores': grade_result['all_scores']
    }

# ── Run on Test Set ────────────────────────────────────────────────────────
test_dir = Path("dataset/unified/test")
all_images = list(test_dir.rglob("*.jpg")) + list(test_dir.rglob("*.jpeg"))

print(f"\nRunning hierarchical classification on {len(all_images)} test images...")

grade_distribution = {'A': 0, 'B': 0, 'C': 0, 'Unknown': 0}
grade_conf_sum = {'A': 0.0, 'B': 0.0, 'C': 0.0}
grade_conf_count = {'A': 0, 'B': 0, 'C': 0}
results_list = []

for img_path in tqdm(all_images):
    img = Image.open(img_path).convert('RGB')
    result = classify_full(img)
    result['path'] = img_path
    results_list.append(result)
    
    g = result['grade']
    grade_distribution[g] = grade_distribution.get(g, 0) + 1
    if g in grade_conf_sum:
        grade_conf_sum[g] += result['grade_confidence']
        grade_conf_count[g] += 1

# ── Grade Report ──────────────────────────────────────────────────────────
Path("results").mkdir(exist_ok=True)

report_lines = [
    "Hierarchical Grade Classification Report",
    "=" * 45,
    f"Total images: {len(all_images)}",
    "",
    "Grade Distribution:",
]
for g in ['A', 'B', 'C', 'Unknown']:
    count = grade_distribution.get(g, 0)
    pct = count / len(all_images) * 100
    avg_conf = (grade_conf_sum.get(g, 0) / grade_conf_count[g] * 100) if grade_conf_count.get(g, 0) > 0 else 0
    action = grade_actions.get(g, 'Flag for human review')
    report_lines.append(f"  Grade {g}: {count:4d} images ({pct:5.1f}%) | Avg confidence: {avg_conf:.1f}% | Action: {action}")

report_lines += [
    "",
    "Grade Descriptions Used:",
    "  Grade A: clean intact plastic in good condition",
    "  Grade B: slightly dirty or mildly damaged plastic",
    "  Grade C: heavily contaminated or severely damaged plastic",
    "",
    "Note: Grade classification uses CLIP zero-shot inference.",
    "      No additional training was required.",
    "      Calibrated for conveyor belt industrial images."
]

report_text = "\n".join(report_lines)
print("\n" + report_text)

with open("results/grade_report.txt", "w") as f:
    f.write(report_text)

# ── Sample Predictions Grid ───────────────────────────────────────────────
sample_results = random.sample(results_list, min(12, len(results_list)))

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle("Hierarchical Classification: Type → Grade", fontsize=16, fontweight='bold')

for ax, res in zip(axes.flat, sample_results):
    img = Image.open(res['path']).convert('RGB').resize((200, 200))
    ax.imshow(img)
    
    grade = res['grade']
    color = grade_colors.get(grade, 'gray')
    
    title = (
        f"Type: {res['type']} ({res['type_confidence']*100:.0f}%)\n"
        f"Grade: {grade} ({res['grade_confidence']*100:.0f}%)\n"
        f"{res['action']}"
    )
    ax.set_title(title, fontsize=7, color=color, fontweight='bold')
    ax.axis('off')
    
    # Colored border by grade
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(3)

plt.tight_layout()
plt.savefig("results/grade_sample_predictions.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/grade_sample_predictions.png")

Device: cuda (Tesla T4)
Loading CLIP ViT-B/32...


AttributeError: module 'clip' has no attribute 'load'

---
## Step 9 — Volumetric Estimation (Depth Anything V2 Metric)

In [ ]:
"""
Step 9: Per-type volumetric estimation using Depth Anything V2 (metric depth).

For each conveyor belt image:
  1. EfficientNet-B3 classifies plastic type per bounding box crop
  2. Depth Anything V2 (metric indoor) estimates real-world depth in meters
  3. Bounding box area × crop depth → volume proxy in cm³

Uses: Depth Anything V2 Metric Indoor Large
Repo: https://github.com/DepthAnything/Depth-Anything-V2/tree/main/metric_depth

Outputs:
  results/volumetric_report.txt           — per-type volume summary
  results/volumetric_annotated/           — conveyor images with volume overlays
  results/volumetric_sample.png           — sample depth map visualization
"""

# ── Install / Import ──────────────────────────────────────────────────────
# Run once if not installed:
# git clone https://github.com/DepthAnything/Depth-Anything-V2
# cd Depth-Anything-V2/metric_depth && pip install -r requirements.txt
# Download checkpoint: depth-anything/Depth-Anything-V2-Metric-Indoor-Large-hf

import sys
import torch
import numpy as np
import cv2
from PIL import Image
from pathlib import Path
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tqdm import tqdm
import json

# Add Depth Anything V2 to path
# Update this path to wherever you cloned the repo
DEPTH_ANYTHING_PATH = "Depth-Anything-V2/metric_depth"
sys.path.insert(0, DEPTH_ANYTHING_PATH)

from depth_anything_v2.dpt import DepthAnythingV2

# ── Device ────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert torch.cuda.is_available(), "GPU required"
print(f"Device: {device} ({torch.cuda.get_device_name(0)})")

# ── Load Depth Anything V2 ────────────────────────────────────────────────
# Model config for Large
model_configs = {
    'vits': {'encoder': 'vits', 'features': 64,  'out_channels': [48,  96,  192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96,  192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
}

ENCODER = 'vitl'  # Large model — most accurate
MAX_DEPTH = 20    # meters — indoor setting

print("Loading Depth Anything V2 Large (metric indoor)...")
depth_model = DepthAnythingV2(**model_configs[ENCODER], max_depth=MAX_DEPTH)

# Update checkpoint path to your downloaded weights
DEPTH_CHECKPOINT = "checkpoints/depth_anything_v2_metric_indoor_vitl.pth"
depth_model.load_state_dict(torch.load(DEPTH_CHECKPOINT, map_location=device))
depth_model.to(device).eval()
print("Depth model loaded")

# ── Load EfficientNet (reuse from Step 8) ─────────────────────────────────
# efficientnet and infer_transform already loaded from Step 8
# If running this cell independently, reload them:
efficientnet = load_efficientnet("checkpoints/phase3_best.pth")

# ── Helper Functions ──────────────────────────────────────────────────────
CONVEYOR_CLASS_MAP = {
    'mix pp':    'PP',
    'mix hd':    'HDPE',
    'mix hdpe':  'HDPE',
    'mix pet':   'PET',
    'mix rigid': 'OTHER',
}

def parse_conveyor_xml(xml_path):
    """Parse Pascal VOC XML, return list of {class, bbox} dicts."""
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        objects = []
        for obj in root.findall('object'):
            name = obj.find('name').text.lower().strip()
            mapped = CONVEYOR_CLASS_MAP.get(name, 'OTHER')
            bbox = obj.find('bndbox')
            objects.append({
                'class': mapped,
                'bbox': (
                    int(float(bbox.find('xmin').text)),
                    int(float(bbox.find('ymin').text)),
                    int(float(bbox.find('xmax').text)),
                    int(float(bbox.find('ymax').text))
                )
            })
        return objects
    except Exception as e:
        print(f"  XML parse error {xml_path}: {e}")
        return []

def estimate_depth_map(cv2_image):
    """Run Depth Anything V2 on a CV2 BGR image, return metric depth in meters."""
    with torch.no_grad():
        depth = depth_model.infer_image(cv2_image)  # returns HxW numpy array in meters
    return depth

def estimate_volume_cm3(bbox, depth_map, image_shape):
    """
    Estimate volume proxy in cm³ from bounding box + metric depth map.
    
    Assumes conveyor belt is at ~1.0m from camera (typical industrial setup).
    Pixel-to-cm conversion uses pinhole camera approximation.
    """
    xmin, ymin, xmax, ymax = bbox
    h, w = image_shape[:2]
    
    # Clamp bbox to image bounds
    xmin, xmax = max(0, xmin), min(w, xmax)
    ymin, ymax = max(0, ymin), min(h, ymax)
    
    if xmax <= xmin or ymax <= ymin:
        return 0.0, 0.0, 0.0
    
    # Crop depth to bounding box region
    bbox_depth = depth_map[ymin:ymax, xmin:xmax]
    avg_depth_m = float(bbox_depth.mean())
    
    # Pixel dimensions of bounding box
    bbox_w_px = xmax - xmin
    bbox_h_px = ymax - ymin
    
    # Approximate real-world dimensions using depth
    # Assuming ~60° FOV camera (standard industrial camera)
    # At depth d meters, each pixel ≈ (d * 2 * tan(30°)) / image_width meters
    import math
    fov_rad = math.radians(60)
    px_to_m = (avg_depth_m * 2 * math.tan(fov_rad / 2)) / w
    
    # Real dimensions in cm
    real_w_cm = bbox_w_px * px_to_m * 100
    real_h_cm = bbox_h_px * px_to_m * 100
    
    # Assume average plastic thickness ~2cm for volume proxy
    PLASTIC_THICKNESS_CM = 2.0
    volume_cm3 = real_w_cm * real_h_cm * PLASTIC_THICKNESS_CM
    
    return volume_cm3, real_w_cm, real_h_cm

# ── Run Volumetric Analysis on Conveyor Dataset ───────────────────────────
conveyor_img_dir = Path("dataset/conveyor/img")
conveyor_lbl_dir = Path("dataset/conveyor/labels")

image_files = sorted(conveyor_img_dir.glob("*.jpeg")) + \
              sorted(conveyor_img_dir.glob("*.jpg"))

print(f"\nRunning volumetric analysis on {len(image_files)} conveyor images...")

Path("results/volumetric_annotated").mkdir(parents=True, exist_ok=True)

# Accumulate per-type volume totals
volume_totals = {}   # class → total cm³
count_totals  = {}   # class → item count
frame_results = []   # per-image summary

for img_path in tqdm(image_files):
    # Find matching XML
    xml_path = conveyor_lbl_dir / (img_path.stem + ".xml")
    if not xml_path.exists():
        continue
    
    annotations = parse_conveyor_xml(xml_path)
    if not annotations:
        continue
    
    # Load image in both formats
    pil_img = Image.open(img_path).convert('RGB')
    cv2_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    
    # Get depth map for full image
    depth_map = estimate_depth_map(cv2_img)
    
    # Annotated image for saving
    annotated = np.array(pil_img).copy()
    
    frame_summary = {'image': img_path.name, 'objects': []}
    
    for ann in annotations:
        plastic_class = ann['class']
        bbox = ann['bbox']
        xmin, ymin, xmax, ymax = bbox
        
        # Volume estimation
        vol_cm3, w_cm, h_cm = estimate_volume_cm3(bbox, depth_map, cv2_img.shape)
        
        # Accumulate
        volume_totals[plastic_class] = volume_totals.get(plastic_class, 0.0) + vol_cm3
        count_totals[plastic_class]  = count_totals.get(plastic_class, 0) + 1
        
        frame_summary['objects'].append({
            'class': plastic_class,
            'volume_cm3': round(vol_cm3, 1),
            'bbox': bbox
        })
        
        # Draw on annotated image
        color_map = {'PET': (0,255,0), 'HDPE': (255,165,0), 
                     'PP': (0,0,255), 'OTHER': (255,0,0)}
        color = color_map.get(plastic_class, (128,128,128))
        cv2.rectangle(annotated, (xmin, ymin), (xmax, ymax), color, 2)
        label = f"{plastic_class} ~{vol_cm3:.0f}cm3"
        cv2.putText(annotated, label, (xmin, max(ymin-5, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
    
    frame_results.append(frame_summary)
    
    # Save annotated image
    save_path = f"results/volumetric_annotated/{img_path.stem}_vol.jpg"
    cv2.imwrite(save_path, cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))

# ── Volumetric Report ─────────────────────────────────────────────────────
report_lines = [
    "Volumetric Estimation Report — Conveyor Belt Dataset",
    "=" * 52,
    f"Images processed : {len(frame_results)}",
    f"Total objects    : {sum(count_totals.values())}",
    "",
    "Per-Type Volume Summary:",
    f"  {'Class':<8} {'Items':>6} {'Total Vol (cm³)':>16} {'Avg Vol/item':>14} {'Approx Liters':>14}",
    "  " + "-" * 62
]

total_vol = sum(volume_totals.values())
for cls in sorted(volume_totals.keys()):
    vol   = volume_totals[cls]
    count = count_totals[cls]
    avg   = vol / count if count > 0 else 0
    liters = vol / 1000
    pct   = vol / total_vol * 100 if total_vol > 0 else 0
    report_lines.append(
        f"  {cls:<8} {count:>6} {vol:>14.0f}   {avg:>12.0f}   {liters:>12.2f}L  ({pct:.1f}%)"
    )

report_lines += [
    "",
    f"  TOTAL    {sum(count_totals.values()):>6} {total_vol:>14.0f}   {'':>12}   {total_vol/1000:>12.2f}L",
    "",
    "Methodology:",
    "  Depth estimation : Depth Anything V2 Large (metric indoor)",
    "  Volume proxy     : bbox_area × avg_depth × 2cm thickness",
    "  Camera model     : Pinhole, 60° FOV assumed",
    "  Units            : cm³ (cubic centimetres)",
    "",
    "Note: Volume is a proxy estimate. True volumetric measurement",
    "requires camera intrinsics calibration for the specific",
    "conveyor belt setup. Relative comparisons between plastic",
    "types are reliable; absolute values require calibration."
]

report_text = "\n".join(report_lines)
print("\n" + report_text)

with open("results/volumetric_report.txt", "w") as f:
    f.write(report_text)

# ── Sample Depth Visualization ────────────────────────────────────────────
# Show one conveyor image + its depth map side by side
sample_path = image_files[0]
sample_cv2 = cv2.imread(str(sample_path))
sample_depth = estimate_depth_map(sample_cv2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.imshow(cv2.cvtColor(sample_cv2, cv2.COLOR_BGR2RGB))
ax1.set_title("Original Conveyor Image", fontweight='bold')
ax1.axis('off')

depth_vis = ax2.imshow(sample_depth, cmap='plasma')
ax2.set_title("Depth Anything V2 — Metric Depth Map\n(meters)", fontweight='bold')
ax2.axis('off')
plt.colorbar(depth_vis, ax=ax2, label='Depth (meters)')

plt.tight_layout()
plt.savefig("results/volumetric_sample.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/volumetric_sample.png")
print("\nVolumetric estimation complete.")

In [36]:
# ─────────────────────────────────────────────────────────────
# STEP 9: FULL Volumetric Estimation (Kaggle FIXED VERSION)
# ─────────────────────────────────────────────────────────────

# ── Install & Setup ──────────────────────────────────────────
!git clone https://github.com/DepthAnything/Depth-Anything-V2.git
!pip install -r Depth-Anything-V2/metric_depth/requirements.txt

import sys
sys.path.insert(0, "/kaggle/working/Depth-Anything-V2/metric_depth")

# ── Imports ──────────────────────────────────────────────────
import torch
import numpy as np
import cv2
from PIL import Image
from pathlib import Path
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from tqdm import tqdm
import math
import json

from depth_anything_v2.dpt import DepthAnythingV2

# ── Device ───────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert torch.cuda.is_available(), "GPU required"
print(f"Device: {device} ({torch.cuda.get_device_name(0)})")

# ── Load Depth Model ─────────────────────────────────────────
DEPTH_CHECKPOINT = "/kaggle/input/YOUR-DATASET/depth_anything_v2_metric_hypersim_vitl.pth"

model_configs = {
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
}

print("Loading Depth Anything V2...")
depth_model = DepthAnythingV2(**model_configs['vitl'], max_depth=20)
depth_model.load_state_dict(torch.load(DEPTH_CHECKPOINT, map_location=device))
depth_model.to(device).eval()
print("Depth model loaded")

# ── Dataset Paths ────────────────────────────────────────────
CONVEYOR_IMG = Path("/kaggle/input/datasets/durvibangera/dataset/conveyor/conveyor/img")
CONVEYOR_LABELS = Path("/kaggle/input/datasets/durvibangera/dataset/conveyor/conveyor/labels")

# ── Helper Functions ─────────────────────────────────────────
CONVEYOR_CLASS_MAP = {
    'mix pp': 'PP',
    'mix hd': 'HDPE',
    'mix hdpe': 'HDPE',
    'mix pet': 'PET',
    'mix rigid': 'OTHER',
}

def parse_xml(xml_path):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        objects = []
        for obj in root.findall('object'):
            name = obj.find('name').text.lower().strip()
            mapped = CONVEYOR_CLASS_MAP.get(name, 'OTHER')
            bbox = obj.find('bndbox')
            objects.append({
                'class': mapped,
                'bbox': (
                    int(float(bbox.find('xmin').text)),
                    int(float(bbox.find('ymin').text)),
                    int(float(bbox.find('xmax').text)),
                    int(float(bbox.find('ymax').text))
                )
            })
        return objects
    except:
        return []

def estimate_depth_map(cv2_img):
    with torch.no_grad():
        return depth_model.infer_image(cv2_img)

def estimate_volume_cm3(bbox, depth_map, image_shape):
    xmin, ymin, xmax, ymax = bbox
    h, w = image_shape[:2]

    xmin, xmax = max(0, xmin), min(w, xmax)
    ymin, ymax = max(0, ymin), min(h, ymax)

    if xmax <= xmin or ymax <= ymin:
        return 0, 0, 0

    depth_crop = depth_map[ymin:ymax, xmin:xmax]
    avg_depth = float(depth_crop.mean())

    bbox_w = xmax - xmin
    bbox_h = ymax - ymin

    px_to_m = (avg_depth * 2 * math.tan(math.radians(60)/2)) / w

    real_w = bbox_w * px_to_m * 100
    real_h = bbox_h * px_to_m * 100

    thickness = 2.0
    volume = real_w * real_h * thickness

    return volume, real_w, real_h

# ── Run Analysis ─────────────────────────────────────────────
image_files = sorted(CONVEYOR_IMG.glob("*.jpg")) + sorted(CONVEYOR_IMG.glob("*.jpeg"))

print(f"\nRunning volumetric analysis on {len(image_files)} images...")

Path("/kaggle/working/results/volumetric_annotated").mkdir(parents=True, exist_ok=True)

volume_totals = {}
count_totals = {}
frame_results = []

for img_path in tqdm(image_files):
    xml_path = CONVEYOR_LABELS / (img_path.stem + ".xml")
    if not xml_path.exists():
        continue

    annotations = parse_xml(xml_path)
    if not annotations:
        continue

    pil_img = Image.open(img_path).convert('RGB')
    cv2_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

    depth_map = estimate_depth_map(cv2_img)
    annotated = np.array(pil_img).copy()

    frame_summary = {'image': img_path.name, 'objects': []}

    for ann in annotations:
        cls = ann['class']
        bbox = ann['bbox']
        xmin, ymin, xmax, ymax = bbox

        vol, w_cm, h_cm = estimate_volume_cm3(bbox, depth_map, cv2_img.shape)

        volume_totals[cls] = volume_totals.get(cls, 0) + vol
        count_totals[cls] = count_totals.get(cls, 0) + 1

        frame_summary['objects'].append({
            'class': cls,
            'volume_cm3': round(vol, 1)
        })

        cv2.rectangle(annotated, (xmin, ymin), (xmax, ymax), (0,255,0), 2)
        cv2.putText(annotated, f"{cls} ~{vol:.0f}cm3",
                    (xmin, max(ymin-5, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0,255,0), 1)

    frame_results.append(frame_summary)

    save_path = f"/kaggle/working/results/volumetric_annotated/{img_path.stem}.jpg"
    cv2.imwrite(save_path, cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))

# ── FULL REPORT ─────────────────────────────────────────────
report_lines = [
    "Volumetric Estimation Report",
    "=" * 40,
    f"Images processed: {len(frame_results)}",
    f"Total objects: {sum(count_totals.values())}",
    "",
]

total_vol = sum(volume_totals.values())

for cls in sorted(volume_totals.keys()):
    vol = volume_totals[cls]
    count = count_totals[cls]
    avg = vol / count if count > 0 else 0
    pct = (vol / total_vol * 100) if total_vol > 0 else 0

    report_lines.append(
        f"{cls}: {count} items | {vol:.0f} cm3 | avg {avg:.0f} | {pct:.1f}%"
    )

report_text = "\n".join(report_lines)
print("\n" + report_text)

with open("/kaggle/working/results/volumetric_report.txt", "w") as f:
    f.write(report_text)

# ── Sample Depth Visualization ───────────────────────────────
sample_path = image_files[0]
sample_cv2 = cv2.imread(str(sample_path))
sample_depth = estimate_depth_map(sample_cv2)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.imshow(cv2.cvtColor(sample_cv2, cv2.COLOR_BGR2RGB))
plt.title("Original")

plt.subplot(1,2,2)
plt.imshow(sample_depth, cmap='plasma')
plt.title("Depth Map")

plt.show()

print("✅ FULL volumetric pipeline complete.")

Cloning into 'Depth-Anything-V2'...
remote: Enumerating objects: 146, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 146 (delta 30), reused 26 (delta 26), pack-reused 84 (from 1)
Receiving objects: 100% (146/146), 45.18 MiB | 40.19 MiB/s, done.
Resolving deltas: 100% (46/46), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 4.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 5.1 MB/s eta 0:00:000:00:0100:01


xFormers not available
xFormers not available


Device: cuda (Tesla T4)


RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-69c492b0-35382a4c611b03c132ca2c70;2e148270-a15d-403d-afc5-13072c23b6cd)

Repository Not Found for url: https://huggingface.co/depth-anything/Depth-Anything-V2-Metric-Indoor-Large/resolve/main/depth_anything_v2_metric_indoor_vitl.pth.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

In [37]:
"""
Step 10a — Convert Pascal VOC XML bounding-box labels to YOLO format.

Reads:  /kaggle/input/datasets/durvibangera/dataset/conveyor/conveyor/img/*.jpeg
        /kaggle/input/datasets/durvibangera/dataset/conveyor/conveyor/labels/*.xml
Writes: /kaggle/working/dataset/yolo_conveyor/
            images/train/  images/val/
            labels/train/  labels/val/
        /kaggle/working/dataset/yolo_conveyor/dataset.yaml

Class map (same 4 plastic types annotated on the conveyor belt):
  0 = HDPE   (mix hd / mix hdpe)
  1 = OTHER  (mix rigid)
  2 = PET    (mix pet)
  3 = PP     (mix pp)
"""

import os, shutil, random, xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict

BASE_DIR        = Path("/kaggle/working")
CONVEYOR_IMG    = Path("/kaggle/input/datasets/durvibangera/dataset/conveyor/conveyor/img")
CONVEYOR_LABELS = Path("/kaggle/input/datasets/durvibangera/dataset/conveyor/conveyor/labels")
YOLO_ROOT       = BASE_DIR / "dataset" / "yolo_conveyor"

YOLO_CLASSES = ["HDPE", "OTHER", "PET", "PP"]
CLASS_TO_IDX = {c: i for i, c in enumerate(YOLO_CLASSES)}

CONVEYOR_CLASS_MAP = {
    "mix pp":    "PP",
    "mix hd":    "HDPE",
    "mix hdpe":  "HDPE",
    "mix pet":   "PET",
    "mix rigid": "OTHER",
    "mix_pp":    "PP",
    "mix_hd":    "HDPE",
    "mix_hdpe":  "HDPE",
    "mix_pet":   "PET",
    "mix_rigid": "OTHER",
}

VAL_RATIO = 0.15
random.seed(42)

# ── collect all labeled images ─────────────────────────────────────────────────
def find_xml(img_stem: str) -> Path | None:
    candidate = CONVEYOR_LABELS / (img_stem + ".xml")
    if candidate.exists():
        return candidate
    for xml in CONVEYOR_LABELS.glob("*.xml"):
        if xml.stem.lower() == img_stem.lower():
            return xml
    return None

def parse_voc_xml(xml_path: Path, img_w: int, img_h: int) -> list[str]:
    """Return YOLO-format label lines: 'cls cx cy w h'"""
    lines = []
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        for obj in root.findall("object"):
            raw = (obj.findtext("name") or "").strip().lower()
            cls_name = CONVEYOR_CLASS_MAP.get(raw)
            if cls_name is None:
                continue
            cls_idx = CLASS_TO_IDX[cls_name]
            bnd = obj.find("bndbox")
            xmin = float(bnd.findtext("xmin"))
            ymin = float(bnd.findtext("ymin"))
            xmax = float(bnd.findtext("xmax"))
            ymax = float(bnd.findtext("ymax"))
            cx = ((xmin + xmax) / 2) / img_w
            cy = ((ymin + ymax) / 2) / img_h
            bw = (xmax - xmin) / img_w
            bh = (ymax - ymin) / img_h
            # clamp
            cx, cy, bw, bh = (min(max(v, 0.0), 1.0) for v in (cx, cy, bw, bh))
            lines.append(f"{cls_idx} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    except Exception as e:
        print(f"  [WARN] failed to parse {xml_path.name}: {e}")
    return lines

img_files = sorted(CONVEYOR_IMG.glob("*.jpg")) + sorted(CONVEYOR_IMG.glob("*.jpeg"))
valid_pairs = []
skipped = 0
for img_path in img_files:
    xml_path = find_xml(img_path.stem)
    if xml_path is None:
        skipped += 1
        continue
    valid_pairs.append((img_path, xml_path))

print(f"Found {len(valid_pairs)} labeled images  ({skipped} skipped — no XML)")

# ── train/val split ────────────────────────────────────────────────────────────
random.shuffle(valid_pairs)
n_val = max(1, int(len(valid_pairs) * VAL_RATIO))
val_pairs   = valid_pairs[:n_val]
train_pairs = valid_pairs[n_val:]
print(f"Split → train: {len(train_pairs)}  val: {len(val_pairs)}")

# ── create folder structure ────────────────────────────────────────────────────
for split in ("train", "val"):
    (YOLO_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

from PIL import Image as PILImage

def write_split(pairs: list, split: str):
    written, empty = 0, 0
    for img_path, xml_path in pairs:
        with PILImage.open(img_path) as im:
            w, h = im.size
        label_lines = parse_voc_xml(xml_path, w, h)
        if not label_lines:
            empty += 1
            continue
        dst_img = YOLO_ROOT / "images" / split / img_path.name
        dst_lbl = YOLO_ROOT / "labels" / split / (img_path.stem + ".txt")
        shutil.copy2(img_path, dst_img)
        dst_lbl.write_text("\n".join(label_lines))
        written += 1
    print(f"  {split}: wrote {written} pairs  ({empty} had no valid boxes)")

write_split(train_pairs, "train")
write_split(val_pairs,   "val")

# ── dataset.yaml ──────────────────────────────────────────────────────────────
yaml_content = f"""path: {YOLO_ROOT.as_posix()}
train: images/train
val:   images/val

nc: {len(YOLO_CLASSES)}
names: {YOLO_CLASSES}
"""
(YOLO_ROOT / "dataset.yaml").write_text(yaml_content)
print(f"\nWrote dataset.yaml → {YOLO_ROOT / 'dataset.yaml'}")
print("Step 10a complete — XML → YOLO conversion done.")


Found 93 labeled images  (2 skipped — no XML)
Split → train: 80  val: 13
  train: wrote 80 pairs  (0 had no valid boxes)
  val: wrote 13 pairs  (0 had no valid boxes)

Wrote dataset.yaml → /kaggle/working/dataset/yolo_conveyor/dataset.yaml
Step 10a complete — XML → YOLO conversion done.


In [38]:
"""
Step 10b — Train YOLOv8n on the conveyor belt dataset.

Uses Ultralytics YOLOv8 nano (fastest to train and smallest to deploy).
With ~80 training images, 100 epochs is sufficient for decent convergence.
Results saved to: /kaggle/working/results/yolo_conveyor/
"""

# Install if needed
import subprocess, sys
try:
    import ultralytics
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics", "-q"])
    import ultralytics

from pathlib import Path
from ultralytics import YOLO

BASE_DIR  = Path("/kaggle/working")
YAML_PATH = BASE_DIR / "dataset" / "yolo_conveyor" / "dataset.yaml"
SAVE_DIR  = BASE_DIR / "results" / "yolo_conveyor"

assert YAML_PATH.exists(), (
    "dataset.yaml not found — run Step 10a first."
)

model = YOLO("yolov8n.pt")   # nano pretrained on COCO — downloads ~6 MB

results = model.train(
    data    = str(YAML_PATH),
    epochs  = 100,
    imgsz   = 640,
    batch   = 8,
    patience= 20,            # early stop if no improvement for 20 epochs
    project = str(SAVE_DIR),
    name    = "train",
    exist_ok= True,
    verbose = True,
    augment = True,          # built-in mosaic + mixup augmentation
    degrees = 15,
    flipud  = 0.5,
    fliplr  = 0.5,
    hsv_h   = 0.015,
    hsv_s   = 0.5,
    hsv_v   = 0.4,
)

best_weights = Path(results.save_dir) / "weights" / "best.pt"
print(f"\nTraining complete.")
print(f"Best weights : {best_weights}")
print(f"Results dir  : {results.save_dir}")

# ── quick validation ───────────────────────────────────────────────────────────
val_metrics = model.val()
print(f"\nmAP50      : {val_metrics.box.map50:.4f}")
print(f"mAP50-95   : {val_metrics.box.map:.4f}")

# ── export to ONNX for deployment ──────────────────────────────────────────────
onnx_path = model.export(format="onnx", imgsz=640, simplify=True)
print(f"\nONNX model : {onnx_path}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.27 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset/yolo_conveyor/dataset.yaml, degrees=15, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torch

In [40]:
"""
Step 10c — Visualise YOLOv8 detections on a sample conveyor image.

Draws predicted bounding boxes + class labels + confidence scores
and saves annotated output to /kaggle/working/results/yolo_sample_detection.png.
"""

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
from ultralytics import YOLO

BASE_DIR     = Path("/kaggle/working")
BEST_PT      = BASE_DIR / "results" / "yolo_conveyor" / "train" / "weights" / "best.pt"
SAMPLE_DIR   = BASE_DIR / "dataset" / "yolo_conveyor" / "images" / "val"
OUT_PATH     = BASE_DIR / "results" / "yolo_sample_detection.png"

assert BEST_PT.exists(), "best.pt not found — run Step 10b first."

model = YOLO(str(BEST_PT))
model.fuse()

# pick the first val image
sample_imgs = sorted(SAMPLE_DIR.glob("*.jpg")) + sorted(SAMPLE_DIR.glob("*.jpeg"))
assert sample_imgs, "No val images found."
sample_path = sample_imgs[0]
print(f"Running detection on: {sample_path.name}")

results = model(str(sample_path), conf=0.25, iou=0.45)[0]

img = Image.open(sample_path).convert("RGB")
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.imshow(img)

COLORS = {"HDPE": "#e74c3c", "OTHER": "#f39c12", "PET": "#2ecc71", "PP": "#3498db"}

for box in results.boxes:
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    conf  = float(box.conf[0])
    cls   = results.names[int(box.cls[0])]
    color = COLORS.get(cls, "#9b59b6")
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=2, edgecolor=color, facecolor="none"
    )
    ax.add_patch(rect)
    ax.text(x1, y1 - 5, f"{cls} {conf:.2f}",
            color=color, fontsize=10, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7))

n = len(results.boxes)
ax.set_title(f"YOLOv8 Detection — {n} object(s) detected", fontsize=14, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig(OUT_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_PATH}")


Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
Running detection on: IN-SURAT-EV-01-03-2025-VU0001_20250310_163220__pp.jpeg

image 1/1 /kaggle/working/dataset/yolo_conveyor/images/val/IN-SURAT-EV-01-03-2025-VU0001_20250310_163220__pp.jpeg: 640x640 6 PPs, 8.1ms
Speed: 1.9ms preprocess, 8.1ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)
Saved: /kaggle/working/results/yolo_sample_detection.png


In [41]:
"""
Step 10d — FastAPI inference server for multi-object plastic detection.

POST /detect   multipart image upload  →  JSON list of detections
GET  /health   liveness check

Run from /kaggle/working/:
    uvicorn detect_api:app --host 0.0.0.0 --port 8000 --reload

Or launch directly from this cell (runs in background thread).
The frontend can POST a FormData with field "file" to http://localhost:8000/detect.

Response JSON example:
    [
      {"label": "PP",   "confidence": 0.87, "box": [120, 45, 310, 280]},
      {"label": "HDPE", "confidence": 0.73, "box": [400, 90, 580, 340]}
    ]
"""

from pathlib import Path

BASE_DIR = Path("/kaggle/working")
BEST_PT  = BASE_DIR / "results" / "yolo_conveyor" / "train" / "weights" / "best.pt"
API_FILE = BASE_DIR / "detect_api.py"

assert BEST_PT.exists(), "best.pt not found — run Step 10b first."

# ── write detect_api.py to /kaggle/working/ ────────────────────────────────────
api_code = f'''"""
FastAPI object-detection endpoint — plastic waste detection via YOLOv8.
Accepts an uploaded image, returns JSON bounding-box detections.
"""

import io
from pathlib import Path
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
from ultralytics import YOLO

MODEL_PATH = Path(r"{BEST_PT}")
app = FastAPI(title="Plastic Waste Detector")

# Allow requests from any origin (adjust for production)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Load model once at startup
_model: YOLO | None = None

@app.on_event("startup")
def load_model():
    global _model
    _model = YOLO(str(MODEL_PATH))
    _model.fuse()
    print(f"Model loaded from {{MODEL_PATH}}")

@app.get("/health")
def health():
    return {{"status": "ok", "model": str(MODEL_PATH.name)}}

@app.post("/detect")
async def detect(file: UploadFile = File(...)):
    if not file.content_type.startswith("image/"):
        raise HTTPException(status_code=400, detail="File must be an image.")
    data = await file.read()
    img = Image.open(io.BytesIO(data)).convert("RGB")
    results = _model(img, conf=0.25, iou=0.45)[0]
    detections = []
    for box in results.boxes:
        x1, y1, x2, y2 = (round(v) for v in box.xyxy[0].tolist())
        detections.append({{
            "label":      results.names[int(box.cls[0])],
            "confidence": round(float(box.conf[0]), 4),
            "box":        [x1, y1, x2, y2],
        }})
    return detections
'''

API_FILE.write_text(api_code)
print(f"Wrote: {API_FILE}")

# ── launch server in a background thread ──────────────────────────────────────
import threading, subprocess, sys, time

def _run_server():
    subprocess.run(
        [sys.executable, "-m", "uvicorn", "detect_api:app",
         "--host", "0.0.0.0", "--port", "8000"],
        cwd=str(BASE_DIR)
    )

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
time.sleep(3)

import urllib.request, json
try:
    with urllib.request.urlopen("http://localhost:8000/health", timeout=5) as r:
        print("Server health:", json.loads(r.read()))
except Exception as e:
    print(f"Server not responding yet ({e}) — wait a moment and retry the health check.")

print("\nEndpoint ready:  POST http://localhost:8000/detect")
print("Frontend usage:  send FormData with field 'file' containing the image.")


Wrote: /kaggle/working/detect_api.py
Server not responding yet (<urlopen error [Errno 111] Connection refused>) — wait a moment and retry the health check.

Endpoint ready:  POST http://localhost:8000/detect
Frontend usage:  send FormData with field 'file' containing the image.


INFO:     Started server process [3805]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
